In [2]:
import os, sys
from pathlib import Path

# ==============================================================================
# Pre-flight (0): 核弹级 PROJECT_ROOT 锚定（防 notebook 路径漂移）
#   - 规则：向上爬直到找到 phycausal_stgrn/ 或 pyproject.toml
#   - 然后强制 os.chdir(PROJECT_ROOT) + sys.path 注入
# ==============================================================================
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "phycausal_stgrn").is_dir():
            return p
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError(
        "Cannot locate PROJECT_ROOT (no phycausal_stgrn/ or pyproject.toml in parents). "
        f"start={start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"✅ [System] cwd locked to PROJECT_ROOT: {PROJECT_ROOT}")

# ==============================================================================
# Pre-flight (1): 核心依赖检查 + CUDA 可用性提示
# ==============================================================================
missing = []
try:
    import torch
    print(f"✅ [Engine] torch={torch.__version__}, cuda_available={torch.cuda.is_available()}")
except Exception:
    missing.append("torch")

try:
    import torchdiffeq
except Exception:
    missing.append("torchdiffeq")


if sys.version_info >= (3, 12):
    print("⚠️ [WARNING] Python >= 3.12 可能导致 PyG/Neural ODE 编译/运行不稳定。建议使用 3.10/3.11。", file=sys.stderr)

if missing:
    print('⚠️ [Preflight] Missing core deps:', ', '.join(missing))
    print('   Please run the next cell to install requirements (pip install).')


✅ [System] cwd locked to PROJECT_ROOT: /home/liangrui/local_mount/PhyCausal-STGRN-demo-plus
⚠️ [Preflight] Missing core deps: torch, torchdiffeq
   Please run the next cell to install requirements (pip install).


In [3]:
# ==============================================================================
# Pre-flight (1): 一键安装/校验依赖（建议首次运行执行）
#   - 你前面遇到: No module named 'scanpy'
#   - 这里会生成一个 requirements_notebook_visiumhd.txt 并按需安装：
#       scanpy, torch, torchdiffeq, pyarrow, fastparquet, h5py, scipy, pandas, numpy
# ==============================================================================
import sys, subprocess, importlib.util
from pathlib import Path

REQ_FILE = Path("requirements_notebook_visiumhd.txt")
REQ_FILE.write_text(
    "\n".join([
        "numpy",
        "pandas",
        "scipy",
        "h5py",
        "torch",
        "torchdiffeq",
        "scanpy",
        "anndata",
        "pyarrow",
        "fastparquet",
    ]) + "\n",
    encoding="utf-8",
)

def _have(pkg: str) -> bool:
    return importlib.util.find_spec(pkg) is not None

missing = []
# 这些 import 名与 pip 包名可能不同，这里按 import 名检查
checks = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("h5py", "h5py"),
    ("torch", "torch"),
    ("torchdiffeq", "torchdiffeq"),
    ("scanpy", "scanpy"),
    ("anndata", "anndata"),
    ("pyarrow", "pyarrow"),
    ("fastparquet", "fastparquet"),
]
for import_name, pip_name in checks:
    if not _have(import_name):
        missing.append(pip_name)

if missing:
    print("⚠️ Missing packages detected:", missing)
    print("Installing via pip -r", str(REQ_FILE))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)])
    print("✅ Install complete. If you still see import errors, restart the kernel once.")
else:
    print("✅ All required packages already available.")


⚠️ Missing packages detected: ['numpy', 'pandas', 'scipy', 'h5py', 'torch', 'torchdiffeq', 'scanpy', 'anndata', 'pyarrow', 'fastparquet']
Installing via pip -r requirements_notebook_visiumhd.txt
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 24.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 34.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 37.0 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 23.8 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 20.4 MB/s  0:00:30m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 35.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 34.6 MB/s  0:00:16m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 42.2 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━

## 真实 Visium HD CRC (16µm bin) — Pre-flight + 训练入口

本 Notebook 已做防御性加固，确保不会重蹈以下覆辙：
- **路径漂移**：强制锁定 `PROJECT_ROOT`，所有 subprocess 显式 `cwd=PROJECT_ROOT`
- **ODE 卡死**：运行前断言 `solver.method=rk4` 且 `use_adjoint=false`（拒跑 dopri5/adjoint）
- **YAML 解包炸裂**：运行前递归扫描，**禁止出现 `options:` 嵌套字典**
- **OOM / 维度炸裂**：真实数据强制 HVG 截断到 64–128，并导出训练可用 `annotation_csv`

### 必备文件（位于：`data/`（本版本假设你把所有相关文件都放在 data/ 下，可递归搜索）

In [4]:

# ==============================================================================
# Pre-flight (2a): 读取 YAML + 断言（防 options 嵌套 / 防 solver kwargs 崩溃 / 防 device 配置失配）
# ==============================================================================
from __future__ import annotations

from pathlib import Path
import yaml
import torch

CFG_CANDIDATES = [
    PROJECT_ROOT / "real_crc_config_nar_gpu_v2.yaml",
    PROJECT_ROOT / "runs" / "real_crc_config_nar_gpu_v2.yaml",
    PROJECT_ROOT / "configs" / "visium_hd_crc_8um.yaml",
    PROJECT_ROOT / "real_crc_config_nar_gpu_balanced.yaml",
    PROJECT_ROOT / "real_crc_config_nar_cpu_balanced.yaml",
    PROJECT_ROOT / "real_crc_config_nar_cpu1h.yaml",
    PROJECT_ROOT / "runs" / "real_crc_config_nar_gpu_balanced.yaml",
    PROJECT_ROOT / "runs" / "real_crc_config_nar_cpu_balanced.yaml",
    PROJECT_ROOT / "runs" / "real_crc_config_nar_cpu1h.yaml",
]
CFG_IN = None
for cand in CFG_CANDIDATES:
    if cand.exists():
        CFG_IN = cand
        break
if CFG_IN is None:
    raise FileNotFoundError("No config found. Checked: " + " | ".join(map(str, CFG_CANDIDATES)))

cfg = yaml.safe_load(CFG_IN.read_text(encoding="utf-8"))
assert isinstance(cfg, dict), f'Config YAML must be a dict, got {type(cfg).__name__}'
cfg.setdefault('data', {})

def assert_no_options(obj, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == "options" or k == "adjoint_options":
                raise TypeError(f"Illegal key '{k}' found at: {path or '<root>'}")
            assert_no_options(v, f"{path}.{k}" if path else str(k))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            assert_no_options(v, f"{path}[{i}]")

assert_no_options(cfg)

solver = cfg.setdefault("solver", {})
if "max_steps" not in solver and "max_nfe" in solver:
    solver["max_steps"] = int(solver.pop("max_nfe"))
if "use_adjoint" not in solver:
    solver["use_adjoint"] = False
if str(solver.get("method", "")).lower() in ("rk4", "euler", "midpoint") and "step_size" not in solver:
    solver["step_size"] = 0.08

if str(cfg.get("device", "cpu")).lower().startswith("cuda") and not torch.cuda.is_available():
    print("⚠️ [Preflight] CUDA requested but unavailable; falling back to CPU.")
    cfg["device"] = "cpu"

print("✅ Loaded config:", CFG_IN)
print("   - device:", cfg.get("device"))
print("   - solver:", {k: solver.get(k) for k in ("method","rtol","atol","max_steps","use_adjoint","step_size") if k in solver})


✅ Loaded config: /home/liangrui/local_mount/PhyCausal-STGRN-demo-plus/runs/real_crc_config_nar_gpu_balanced.yaml
   - device: cuda
   - solver: {'method': 'rk4', 'rtol': 0.001, 'atol': 0.001, 'max_steps': 96, 'use_adjoint': False, 'step_size': 0.08}


In [ ]:
# ==============================================================================
# Pre-flight (2a+): claim / pairing 开关（不同数据集可手动切换）
#   - main_mechanism: 主数据集，默认 OT=off, dynamic_anchor=off
#   - ot_pairing_validation: OT 配对验证数据集，默认 OT=on
#   - dynamic_anchor_validation: 动态锚点验证数据集，默认 dynamic_anchor=on
#   - full_claims: 两者都开（仅在环境与数据都准备好时使用）
# ==============================================================================
from __future__ import annotations

from pprint import pprint

CLAIM_PROFILE = globals().get("CLAIM_PROFILE", "main_mechanism")

# 允许用户直接改这两个布尔值；若设为 None，则按 CLAIM_PROFILE 默认值
USER_ENABLE_OT_PAIRING = globals().get("USER_ENABLE_OT_PAIRING", None)
USER_ENABLE_DYNAMIC_ANCHOR_PAIRING = globals().get("USER_ENABLE_DYNAMIC_ANCHOR_PAIRING", None)

_CLAIM_PRESETS = {
    "main_mechanism": {
        "enable_ot_pairing": False,
        "enable_dynamic_anchor_pairing": False,
        "description": "主数据集：机制主结论优先，OT/动态锚点默认关闭。",
    },
    "ot_pairing_validation": {
        "enable_ot_pairing": True,
        "enable_dynamic_anchor_pairing": False,
        "description": "OT 配对验证数据集：开启 OT，关闭动态锚点。",
    },
    "dynamic_anchor_validation": {
        "enable_ot_pairing": False,
        "enable_dynamic_anchor_pairing": True,
        "description": "动态锚点验证数据集：关闭 OT，开启动态锚点。",
    },
    "full_claims": {
        "enable_ot_pairing": True,
        "enable_dynamic_anchor_pairing": True,
        "description": "全开模式：仅在依赖和数据都齐全时使用。",
    },
}

if CLAIM_PROFILE not in _CLAIM_PRESETS:
    raise ValueError(f"Unknown CLAIM_PROFILE={CLAIM_PROFILE!r}. Choices: {sorted(_CLAIM_PRESETS)}")

_claim_runtime = dict(_CLAIM_PRESETS[CLAIM_PROFILE])
if USER_ENABLE_OT_PAIRING is not None:
    _claim_runtime["enable_ot_pairing"] = bool(USER_ENABLE_OT_PAIRING)
if USER_ENABLE_DYNAMIC_ANCHOR_PAIRING is not None:
    _claim_runtime["enable_dynamic_anchor_pairing"] = bool(USER_ENABLE_DYNAMIC_ANCHOR_PAIRING)

ENABLE_OT_PAIRING = bool(_claim_runtime["enable_ot_pairing"])
ENABLE_DYNAMIC_ANCHOR_PAIRING = bool(_claim_runtime["enable_dynamic_anchor_pairing"])

print("✅ [Claim Control] Active profile:", CLAIM_PROFILE)
pprint({
    "description": _claim_runtime["description"],
    "ENABLE_OT_PAIRING": ENABLE_OT_PAIRING,
    "ENABLE_DYNAMIC_ANCHOR_PAIRING": ENABLE_DYNAMIC_ANCHOR_PAIRING,
})

# 供后续 cell 直接使用的统一元数据
CLAIM_RUNTIME = {
    "claim_profile": CLAIM_PROFILE,
    "enable_ot_pairing": ENABLE_OT_PAIRING,
    "enable_dynamic_anchor_pairing": ENABLE_DYNAMIC_ANCHOR_PAIRING,
    "user_override_ot_pairing": USER_ENABLE_OT_PAIRING,
    "user_override_dynamic_anchor_pairing": USER_ENABLE_DYNAMIC_ANCHOR_PAIRING,
}


In [5]:
# ==============================================================================
# Pre-flight (2b): 自动定位文件（优先 GSM8594569_P5CRC_filtered_feature_bc_matrix.h5）
#   约定：你说“所有数据都放在 data/ 下”，因此这里对 data/ 递归搜索。
# ==============================================================================
from __future__ import annotations

from pathlib import Path
from typing import Sequence, Tuple, List, Optional

import os

# ------------------------------------------------------------------------------
# DATA_DIR 解析：优先从 cfg/YAML 读取，其次环境变量，其次默认 PROJECT_ROOT/data
# ------------------------------------------------------------------------------
if "DATA_DIR" not in globals():
    _cand = None
    _cfg = globals().get("cfg", {})
    try:
        if isinstance(_cfg, dict):
            for _k in ["DATA_DIR", "data_dir", "data_path", "root", "base_dir", "input_dir"]:
                if _k in _cfg and _cfg[_k]:
                    _cand = _cfg[_k]
                    break
            if _cand is None:
                _d = _cfg.get("data", {})
                if isinstance(_d, dict):
                    for _k in ["DATA_DIR", "data_dir", "data_path", "root", "base_dir", "input_dir", "dir"]:
                        if _k in _d and _d[_k]:
                            _cand = _d[_k]
                            break
    except Exception as _e:
        print(f"⚠️ [Preflight] cfg parse for DATA_DIR failed: {_e}")

    if _cand is None:
        _cand = os.environ.get("DATA_DIR") or os.environ.get("VISIUM_DATA_DIR")

    if _cand is None:
        _root = globals().get("PROJECT_ROOT", Path.cwd())
        for _rel in ["data", "dataset", "datasets", "crc_data"]:
            _p = Path(_root) / _rel
            if _p.exists():
                _cand = _p
                break

    if _cand is None:
        _cand = Path(globals().get("PROJECT_ROOT", Path.cwd())) / "data"

    DATA_DIR = Path(_cand).expanduser().resolve()

print(f"✅ [Preflight] DATA_DIR = {DATA_DIR}")

SAMPLE_TAG = "GSM8594569_P5CRC"
BIN_TAG = os.environ.get("PHYCAUSAL_BIN_TAG", "8um")  # 默认按 8um 运行；若使用其他 bin，可用环境变量覆盖

def rglob_all(base: Path, patterns: Sequence[str]) -> List[Path]:
    out: List[Path] = []
    for pat in patterns:
        out.extend(base.rglob(pat))
    # 去重
    seen: set[str] = set()
    uniq: List[Path] = []
    for p in out:
        rp = str(p.resolve())
        if rp not in seen:
            seen.add(rp)
            uniq.append(p)
    return uniq

def pick_best(paths: Sequence[Path], prefer_tokens: Sequence[str]) -> Tuple[Optional[Path], List[Path]]:
    """Return (best_path_or_None, candidates_list). Always returns a list (never None)."""
    if not paths:
        return None, []
    def score(p: Path) -> tuple[int, int]:
        name = p.name.lower()
        return (sum(1 for t in prefer_tokens if t.lower() in name), -len(name))
    cands = sorted(list(paths), key=score, reverse=True)
    return cands[0], cands

# 1) h5 matrix
h5_exact = DATA_DIR / f"{SAMPLE_TAG}_filtered_feature_bc_matrix.h5"
if h5_exact.exists():
    h5_path: Path = h5_exact
    h5_cands: List[Path] = [h5_exact]
else:
    h5_path, h5_cands = pick_best(
        rglob_all(DATA_DIR, ["*filtered_feature_bc_matrix.h5", "*feature_bc_matrix.h5", "*.h5"]),
        prefer_tokens=[SAMPLE_TAG, "filtered_feature_bc_matrix", "feature_bc_matrix"],
    )

if h5_path is None:
    raise FileNotFoundError(f"Cannot find 10x .h5 matrix under: {DATA_DIR}")

# 2) tissue positions (parquet/csv, 支持 .gz)
pos_path, pos_cands = pick_best(
    rglob_all(DATA_DIR, ["*tissue_positions*.parquet*", "*positions*.parquet*", "*tissue_positions*.csv*", "*positions*.csv*"]),
    prefer_tokens=[SAMPLE_TAG, BIN_TAG, "tissue_positions"],
)
if pos_path is None:
    raise FileNotFoundError(f"Cannot find tissue positions under: {DATA_DIR}")

# 3) annotations (只匹配文件名含 annotation，避免误抓 sim_gold_grn.csv 等)
ann_path, ann_cands = pick_best(
    rglob_all(DATA_DIR, ["*annotation*.csv*", "*annotations*.csv*", "*squares_annotation*.csv*"]),
    prefer_tokens=[BIN_TAG, SAMPLE_TAG, "annotation"],
)
if ann_path is None:
    raise FileNotFoundError(
        f"Cannot find annotation csv under: {DATA_DIR}\n"
        "Please ensure file name contains 'annotation', e.g. 8um_squares_annotation.csv"
    )

if len(ann_cands) > 1:
    print(f"⚠️ [Preflight] Multiple annotation candidates found; auto-selected: {ann_path}")
    print("  Candidates:")
    for p in ann_cands:
        print("   -", p)

print("✅ [Preflight] Found files:")
print("  - h5 :", h5_path)
print("  - pos:", pos_path)
print("  - ann:", ann_path)


✅ [Preflight] DATA_DIR = /home/liangrui/local_mount/PhyCausal-STGRN-demo-plus/data
⚠️ [Preflight] Multiple annotation candidates found; auto-selected: /home/liangrui/local_mount/PhyCausal-STGRN-demo-plus/data/16um_squares_annotation.csv
  Candidates:
   - /home/liangrui/local_mount/PhyCausal-STGRN-demo-plus/data/16um_squares_annotation.csv
   - /home/liangrui/local_mount/PhyCausal-STGRN-demo-plus/data/8um_squares_annotation.csv
✅ [Preflight] Found files:
  - h5 : /home/liangrui/local_mount/PhyCausal-STGRN-demo-plus/data/GSM8594569_P5CRC_filtered_feature_bc_matrix.h5
  - pos: /home/liangrui/local_mount/PhyCausal-STGRN-demo-plus/data/GSM8594569_P5CRC_tissue_positions.parquet.gz
  - ann: /home/liangrui/local_mount/PhyCausal-STGRN-demo-plus/data/16um_squares_annotation.csv


In [ ]:
# --- Sync BIN_TAG with the actual selected annotation path ---
from pathlib import Path
import os

_ann_path = globals().get("ANN_CSV", globals().get("annotation_path", globals().get("ann_path", None)))
if _ann_path is not None:
    _ann_name = str(_ann_path).lower()
    if "8um" in _ann_name or "008um" in _ann_name:
        BIN_TAG = "8um"
    elif "16um" in _ann_name or "016um" in _ann_name:
        BIN_TAG = "16um"
    print(f"✅ [BIN_TAG] synced to {BIN_TAG} from annotation path: {_ann_path}")
else:
    BIN_TAG = globals().get("BIN_TAG", os.environ.get("PHYCAUSAL_BIN_TAG", "8um"))
    print(f"⚠️ [BIN_TAG] annotation path not found in globals; keep BIN_TAG={BIN_TAG}")


In [6]:
# ==============================================================================
# Pre-flight (2c): 读取矩阵（优先 scanpy；否则 h5py fallback）
#   并保留 raw/normalized barcode，用于后续自动匹配 positions/annotations。
# ==============================================================================
from __future__ import annotations

from typing import List, Tuple, Optional, cast, Any
import numpy as np
import pandas as pd
import scipy.sparse as sp

def normalize_bc(arr) -> np.ndarray:
    """
    Barcode normalization that is *safe* for Visium HD binned barcodes.

    - For classic 10x barcodes: 'AAAC...-1' -> strip trailing '-1' (or '-2', etc.)
    - For Visium HD binned barcodes: 's_016um_00052_00082-1' -> strip ONLY the trailing '-1'
      (DO NOT strip the '_00052_00082' coordinate parts).
    """
    s = pd.Series(arr, dtype="string").astype("string").str.strip()

    # Detect Visium-HD binned barcode style (starts with s_ and contains _um_)
    # Example: s_016um_00052_00082-1
    is_visium_bin = s.str.match(r"^s_\d+um_\d+_\d+(?:-\d+)?$", na=False).mean() > 0.2

    # In both cases we ONLY strip trailing '-<digits>' groups, never underscores.
    s = s.str.replace(r"(-\d+)+$", "", regex=True)

    return s.astype(str).to_numpy()

# --- Try scanpy first (更可靠)
_USING_SCANPY: bool
try:
    import scanpy as sc  # type: ignore
    adata = sc.read_10x_h5(str(h5_path))
    adata.var_names_make_unique()

    matrix_barcodes_raw: List[str] = list(map(str, adata.obs_names))
    matrix_barcodes_norm_arr = normalize_bc(matrix_barcodes_raw)
    matrix_barcodes_norm: List[str] = list(map(str, matrix_barcodes_norm_arr))

    # 统一 barcode（使用 norm 作为主键）
    # 用 Index(str) 让静态检查器满意
    adata.obs_names = pd.Index(matrix_barcodes_norm, dtype="string")

    print(f"✅ [Preflight] Loaded matrix via scanpy: n_spots={adata.n_obs}, n_genes_raw={adata.n_vars}")
    print("   - barcode sample (raw -> norm):")
    for a, b in list(zip(matrix_barcodes_raw, matrix_barcodes_norm))[:3]:
        print("     ", a, "->", b)

    _USING_SCANPY = True
except Exception as e:
    print("⚠️ [Preflight] scanpy load failed, falling back to h5py. Error:", type(e).__name__, str(e)[:200])
    _USING_SCANPY = False

# --- h5py fallback
if not _USING_SCANPY:
    import h5py  # type: ignore
    from pathlib import Path

    def _decode_bytes(arr) -> List[str]:
        out: List[str] = []
        for b in arr:
            if isinstance(b, (bytes, bytearray)):
                out.append(b.decode("utf-8", errors="ignore"))
            else:
                out.append(str(b))
        return out

    def read_10x_h5_basic(path: Path) -> Tuple[sp.csr_matrix, List[str], List[str], List[str]]:
        with h5py.File(path, "r") as f:
            if "matrix" not in f:
                raise ValueError("Unsupported 10x h5 structure: missing /matrix group")

            m = cast(Any, f["matrix"])  # h5py stubs are incomplete; treat as Any for static checking
            data = np.asarray(m["data"])
            indices = np.asarray(m["indices"])
            indptr = np.asarray(m["indptr"])
            shape_arr = np.asarray(m["shape"])
            shape = (int(shape_arr[0]), int(shape_arr[1]))

            X_csc = sp.csc_matrix((data, indices, indptr), shape=shape, dtype=np.float32)
            X: sp.csr_matrix = X_csc.tocsr()

            barcodes_raw = _decode_bytes(np.asarray(m["barcodes"]))
            barcodes_norm = list(map(str, normalize_bc(barcodes_raw)))

            features = cast(Any, m["features"])  # type: ignore[assignment]
            if "name" in features.keys():
                names = np.asarray(features["name"])
            elif "gene_names" in features.keys():
                names = np.asarray(features["gene_names"])
            else:
                raise KeyError("Unsupported 10x h5: /matrix/features missing 'name' or 'gene_names'")
            genes = _decode_bytes(names)

            return X, barcodes_raw, barcodes_norm, genes

    X_raw, matrix_barcodes_raw, matrix_barcodes_norm, genes = read_10x_h5_basic(Path(h5_path))
    print(f"✅ [Preflight] Loaded matrix via h5py: n_spots={X_raw.shape[0]}, n_genes_raw={X_raw.shape[1]}")
    print("   - barcode sample (raw -> norm):")
    for a, b in list(zip(matrix_barcodes_raw, matrix_barcodes_norm))[:3]:
        print("     ", a, "->", b)


# --- compatibility vars for downstream / legacy cells -------------------------
# Some downstream cells (or stale in-memory notebook states) may expect X_csr / gene_names.
# Define them consistently for both scanpy and h5py paths.
if _USING_SCANPY:
    X_csr = sp.csr_matrix(adata.X) if not sp.issparse(adata.X) else adata.X.tocsr()
    gene_names = adata.var_names.astype(str).tolist()
else:
    X_csr = X_raw.tocsr() if sp.issparse(X_raw) else sp.csr_matrix(X_raw)
    gene_names = list(map(str, genes))
    try:
        import anndata as ad  # type: ignore
        adata = ad.AnnData(X=X_csr)
        adata.obs_names = pd.Index(matrix_barcodes_norm, dtype="string")
        adata.var_names = pd.Index(gene_names, dtype="string")
    except Exception:
        # Keep X_csr/gene_names available even if AnnData construction fails.
        pass

print(f"✅ [Preflight] Matrix compatibility vars ready: X_csr={X_csr.shape}, gene_names={len(gene_names)}")


/home/liangrui/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/liangrui/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


✅ [Preflight] Loaded matrix via scanpy: n_spots=541968, n_genes_raw=18085
   - barcode sample (raw -> norm):
      s_008um_00602_00290-1 -> s_008um_00602_00290
      s_008um_00789_00234-1 -> s_008um_00789_00234
      s_008um_00728_00006-1 -> s_008um_00728_00006


In [7]:
# ==============================================================================
# Pre-flight (2d): 读取 positions / annotation（支持 parquet.gz + 编码/分隔符鲁棒）
#   并提供 load_positions/load_annotation 用于“自动找能对齐 barcode 的文件”。
# ==============================================================================
from __future__ import annotations

import gzip, shutil, tempfile, os
import pandas as pd
from typing import Sequence, Tuple, Optional, List, Any
from pathlib import Path

_ANN_WARNED: set[str] = set()

def _read_head_text(path: Path, nbytes: int = 4096) -> str:
    """Read a small text header from plain or gz file (best-effort)."""
    try:
        if path.name.lower().endswith(".gz"):
            with gzip.open(path, "rt", errors="ignore") as f:
                return f.read(nbytes)
        else:
            with open(path, "rt", errors="ignore") as f:
                return f.read(nbytes)
    except Exception:
        return ""

def robust_read_table(path: Path) -> pd.DataFrame:
    """
    Robustly read a delimited text table (csv/tsv, possibly .gz).
    Handles:
      - unknown encoding (utf-8/utf-8-sig/gbk/cp936/latin1)
      - unknown delimiter (comma / tab / semicolon)
      - files without header (first row is data)
    """
    head = _read_head_text(path)
    head_low = head.lower()
    has_tab = "\t" in head
    has_comma = "," in head
    has_semi = ";" in head

    encodings = ("utf-8", "utf-8-sig", "gbk", "cp936", "latin1")
    likely_no_header = ("barcode" not in head_low) and ("barcodes" not in head_low)

    # 1) sniffer-based
    for enc in encodings:
        try:
            df = pd.read_csv(
                path,
                encoding=enc,
                compression="infer",
                sep=None,
                engine="python",
                header=None if likely_no_header else "infer",
            )
            if df.shape[1] == 1 and has_tab:
                raise ValueError("Likely TSV; retry with sep='\\t'")
            if df.columns.dtype == "int64" or all(isinstance(c, int) for c in df.columns):
                df.columns = [f"col_{i}" for i in range(df.shape[1])]
            return df
        except Exception:
            continue

    # 2) explicit delimiter tries
    delimiter_trials = []
    if has_tab: delimiter_trials.append("\t")
    if has_comma: delimiter_trials.append(",")
    if has_semi: delimiter_trials.append(";")
    delimiter_trials += ["\t", ",", ";"]

    for enc in encodings:
        for sep in delimiter_trials:
            header_orders = [None, "infer"] if likely_no_header else ["infer", None]
            for hdr in header_orders:
                try:
                    df = pd.read_csv(path, encoding=enc, compression="infer", sep=sep, header=hdr)
                    if df.shape[1] >= 2:
                        if hdr is None:
                            df.columns = [f"col_{i}" for i in range(df.shape[1])]
                        return df
                except Exception:
                    pass

    # 3) last resort (pandas>=2)
    try:
        df = pd.read_csv(
            path,
            encoding="utf-8",
            compression="infer",
            sep=None,
            engine="python",
            header=None if likely_no_header else "infer",
            encoding_errors="replace",
        )
        if df.columns.dtype == "int64" or all(isinstance(c, int) for c in df.columns):
            df.columns = [f"col_{i}" for i in range(df.shape[1])]
        return df
    except Exception as e:
        raise RuntimeError(
            f"Failed to read table: {path}\n"
            f"Head preview (first 2k chars):\n{head[:2000]}\n"
            f"Error: {type(e).__name__}: {e}"
        )

def read_parquet_maybe_gz(path: Path) -> pd.DataFrame:
    """Read parquet (optionally .gz). If file is mislabeled (e.g., .parquet.gz but actually CSV/TSV),
    automatically fall back to robust_read_table instead of hard failing.

    This fixes the common Visium/SpaceRanger packaging issue where tissue_positions is shipped as
    a gzipped CSV but renamed to .parquet.gz.
    """
    # Try direct parquet read first
    try:
        return pd.read_parquet(path)
    except Exception as e1:
        # gzip-wrapped parquet: .parquet.gz
        if path.name.lower().endswith(".parquet.gz"):
            tmp_path = None
            try:
                with gzip.open(path, "rb") as f_in, tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
                    shutil.copyfileobj(f_in, tmp)
                    tmp_path = tmp.name

                # Try parquet on decompressed payload
                try:
                    df = pd.read_parquet(tmp_path)
                    return df
                except Exception as e2:
                    # Fallback: payload might be text table (csv/tsv) rather than parquet.
                    head = _read_head_text(Path(tmp_path), nbytes=2048)
                    if any(k in head.lower() for k in ["barcode", "barcodes", "pxl", "array_row", "array_col", "tissue"]):
                        try:
                            df = robust_read_table(Path(tmp_path))
                            return df
                        except Exception:
                            pass
                    # If header looks like plain text (has commas/tabs and multiple lines), try as table anyway
                    if ("\n" in head) and (("," in head) or ("\t" in head) or (";" in head)):
                        try:
                            df = robust_read_table(Path(tmp_path))
                            return df
                        except Exception:
                            pass

                    raise RuntimeError(
                        f"Failed to read gzip-wrapped parquet: {path}\n"
                        "该文件可能不是 parquet（常见：其实是 gzipped CSV/TSV 但后缀写成 .parquet.gz）。\n"
                        "你可以：1) 把文件改名为 .csv.gz；或 2) 让 notebook 自动 fallback（已尝试）。\n"
                        f"First error: {type(e1).__name__}: {e1}\n"
                        f"Second error: {type(e2).__name__}: {e2}"
                    )
            finally:
                if tmp_path and os.path.exists(tmp_path):
                    try:
                        os.unlink(tmp_path)
                    except Exception:
                        pass

        # Non-gz parquet failed: maybe mislabeled text
        head = _read_head_text(path, nbytes=2048)
        if ("\n" in head) and (("," in head) or ("\t" in head) or (";" in head)):
            try:
                return robust_read_table(path)
            except Exception:
                pass

        raise RuntimeError(
            f"Failed to read parquet: {path}\n"
            "请确保这是 parquet 文件，或改成 .csv/.tsv（如果其实是文本表）。\n"
            f"Original error: {type(e1).__name__}: {e1}"
        )


def safe_read_positions(path: Path) -> pd.DataFrame:
    name = path.name.lower()
    if name.endswith(".parquet") or name.endswith(".pq") or name.endswith(".parquet.gz") or ".parquet" in "".join(path.suffixes).lower():
        return read_parquet_maybe_gz(path)
    if name.endswith(".csv") or name.endswith(".csv.gz") or name.endswith(".tsv") or name.endswith(".tsv.gz"):
        return robust_read_table(path)
    return robust_read_table(path)

def infer_col(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[object]:
    low = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    return None

def load_positions(path: Path):
    df = safe_read_positions(path)
    pos_bc = infer_col(df, ["barcode", "barcodes", "spot_id", "id", "key"])
    pos_x  = infer_col(df, ["x", "pxl_col_in_fullres", "center_x", "col", "array_col"])
    pos_y  = infer_col(df, ["y", "pxl_row_in_fullres", "center_y", "row", "array_row"])
    if pos_bc is None or pos_x is None or pos_y is None:
        raise KeyError(f"Cannot infer required columns in positions. Columns={list(df.columns)}")
    assert pos_bc is not None and pos_x is not None and pos_y is not None
    df = df.copy()
    # normalize + strip
    df[pos_bc] = normalize_bc(df[pos_bc].astype(str).values)
    df[pos_bc] = pd.Series(df[pos_bc]).astype(str).str.strip().values
    df = df.drop_duplicates(subset=[pos_bc]).set_index(pos_bc)
    return df, pos_x, pos_y, pos_bc

def load_annotation(path: Path):
    df = robust_read_table(path)
    ann_bc  = infer_col(df, ["barcode", "barcodes", "spot_id", "id", "key"])
    ann_lab = infer_col(df, ["annotation", "annotations", "label", "cell_type", "celltype", "region", "pathology"])
    if (ann_bc is None or ann_lab is None) and df.shape[1] >= 2:
        ann_bc = df.columns[0]
        ann_lab = df.columns[1]
        if str(path) not in _ANN_WARNED:
            print("⚠️ [Preflight] Annotation columns not named; using first two columns as (barcode, label).")
            _ANN_WARNED.add(str(path))
    if ann_bc is None or ann_lab is None:
        raise KeyError(
            "Cannot infer required columns in annotation.\n"
            f"Columns={list(df.columns)}\n"
            "Tip: your annotation file may be TSV without header. Ensure it has >=2 columns: barcode + label."
        )
    assert ann_bc is not None and ann_lab is not None
    df = df.copy()
    df[ann_bc] = normalize_bc(df[ann_bc].astype(str).values)
    df[ann_bc] = pd.Series(df[ann_bc]).astype(str).str.strip().values
    df[ann_lab] = df[ann_lab].astype(str).str.strip()
    df = df.drop_duplicates(subset=[ann_bc]).set_index(ann_bc)
    return df, ann_lab, ann_bc

# Load selected paths (may be auto-corrected in next cell if mismatch)
pos, pos_x, pos_y, pos_bc = load_positions(Path(pos_path))
ann, ann_lab, ann_bc = load_annotation(Path(ann_path))

print(f"✅ [Preflight] Loaded positions: {pos.shape}, annotation: {ann.shape}")
print(f"   - positions cols: bc={pos_bc}, x={pos_x}, y={pos_y}")
print(f"   - annotation cols: bc={ann_bc}, label={ann_lab}")


⚠️ [Preflight] Annotation columns not named; using first two columns as (barcode, label).
✅ [Preflight] Loaded positions: (702244, 5), annotation: (137051, 1)
   - positions cols: bc=barcode, x=pxl_col_in_fullres, y=pxl_row_in_fullres
   - annotation cols: bc=col_0, label=col_1


In [8]:

# ==============================================================================
# Pre-flight (2e): 对齐 barcode + HVG(64~128) + env_map 校验
#   若无重叠：自动在候选 pos/ann 文件中选择“能与矩阵 barcode 对齐”的组合。
#   注意：这里只做 common-spot 对齐与 annotation / spatial 挂载，不在这里做 HVG 截断。
# ==============================================================================
from __future__ import annotations

from pathlib import Path
from typing import List, Optional, Tuple, Any, cast

import numpy as np
import pandas as pd
import scipy.sparse as sp


def _obs_variants() -> List[Tuple[str, pd.Index]]:
    """Provide multiple matrix barcode variants to maximize overlap."""
    raw = pd.Index(matrix_barcodes_raw, name="barcode").astype(str).str.strip()
    norm = pd.Index(matrix_barcodes_norm, name="barcode").astype(str).str.strip()
    norm_m1 = pd.Index((pd.Series(matrix_barcodes_norm, dtype="string") + "-1").astype(str).values, name="barcode")
    return [("raw", raw), ("norm", norm), ("norm_plus_-1", norm_m1)]


def _pick_best_combo(
    pos_candidates: List[Path],
    ann_candidates: List[Path],
) -> Optional[Tuple[int, Path, Path, str, pd.Index, pd.DataFrame, str, str, pd.DataFrame, str]]:
    best: Optional[Tuple[int, Path, Path, str, pd.Index, pd.DataFrame, str, str, pd.DataFrame, str]] = None
    for obs_name, obs_idx in _obs_variants():
        for pp in pos_candidates:
            try:
                pos_df, _pos_x, _pos_y, _pos_bc = load_positions(pp)
            except Exception:
                continue
            if len(pos_df.index) == 0:
                continue

            for ap in ann_candidates:
                try:
                    ann_df, _ann_lab, _ann_bc = load_annotation(ap)
                except Exception:
                    continue
                if len(ann_df.index) == 0:
                    continue

                common_idx = obs_idx.intersection(pos_df.index).intersection(ann_df.index)
                score_here = int(len(common_idx))
                if best is None or score_here > best[0]:
                    best = (score_here, pp, ap, obs_name, obs_idx, pos_df, _pos_x, _pos_y, ann_df, _ann_lab)
    return best


def _infer_bin_hint(idx: pd.Index) -> Optional[str]:
    s = pd.Series(idx.astype(str)).astype("string").str.lower()
    if (s.str.contains("016um") | s.str.contains("16um")).mean() > 0.05:
        return "16um"
    if (s.str.contains("008um") | s.str.contains("8um")).mean() > 0.05:
        return "8um"
    return None


obs_default = pd.Index(matrix_barcodes_norm, name="barcode").astype(str).str.strip()

# Default bindings: keep current files unless auto-match proves better
obs = obs_default
score = int(len(obs_default.intersection(pos.index).intersection(ann.index)))
best_pos_path = cast(Path, pos_path)
best_ann_path = cast(Path, ann_path)
best_pos = pos
best_pos_x = pos_x
best_pos_y = pos_y
best_ann = ann
best_ann_lab = ann_lab
common = obs.intersection(best_pos.index).intersection(best_ann.index)

if len(common) == 0:
    print("⚠️ [Preflight] No overlapping barcodes with current (pos_path, ann_path). Trying candidates auto-match...")

    pos_candidates: List[Path] = list(pos_cands) if isinstance(pos_cands, list) and len(pos_cands) > 0 else [cast(Path, pos_path)]
    ann_candidates: List[Path] = list(ann_cands) if isinstance(ann_cands, list) and len(ann_cands) > 0 else [cast(Path, ann_path)]

    _bin_hint = _infer_bin_hint(obs_default)
    if _bin_hint is not None:
        ann_filt = [p for p in ann_candidates if _bin_hint in str(p).lower()]
        if len(ann_filt) > 0:
            ann_candidates = ann_filt

    best = _pick_best_combo(pos_candidates, ann_candidates)
    if best is None or best[0] == 0:
        print("❌ [Preflight] Auto-match failed: still no overlapping barcodes.")
        print("   - matrix barcode sample:", list(obs_default[:5]))
        print("   - positions barcode sample:", list(pos.index.astype(str)[:5]))
        print("   - annotation barcode sample:", list(ann.index.astype(str)[:5]))
        print("   - pos candidates (top10):", [str(p) for p in pos_candidates[:10]])
        print("   - ann candidates (top10):", [str(p) for p in ann_candidates[:10]])
        raise RuntimeError(
            "No overlapping barcodes among (h5 matrix, positions, annotations) after auto-matching.\n"
            "Most common causes:\n"
            "  1) You are mixing different bin sizes (8um vs 16um) between matrix/positions/annotation.\n"
            "  2) The .h5 is not the binned matrix corresponding to the annotation (e.g., raw 10x barcodes vs s_016um_*).\n"
            "Fix: ensure positions + annotation are generated for the SAME binning as the .h5 matrix."
        )

    score, best_pos_path, best_ann_path, obs_name, obs_idx, best_pos, best_pos_x, best_pos_y, best_ann, best_ann_lab = best
    min_required = max(1000, int(0.001 * len(obs_idx)))
    if score < min_required:
        raise RuntimeError(
            f"Auto-matched overlap too small (common={score}, required>={min_required}).\n"
            "This usually means your matrix/positions/annotation are from DIFFERENT bin sizes (8um vs 16um) "
            "or barcode normalization is still incorrect.\n"
            f"Chosen positions: {best_pos_path}\n"
            f"Chosen annotation: {best_ann_path}\n"
            "Fix: ensure the .h5, tissue_positions, and annotation come from the SAME binned output."
        )

    print(f"✅ [Preflight] Auto-matched best combo with common={score}")
    print(f"   - obs variant : {obs_name}")
    print(f"   - positions   : {best_pos_path}")
    print(f"   - annotation  : {best_ann_path}")

    # overwrite globals for downstream
    pos_path = best_pos_path
    ann_path = best_ann_path
    pos = best_pos
    pos_x, pos_y = best_pos_x, best_pos_y
    ann = best_ann
    ann_lab = best_ann_lab
    obs = obs_idx
    common = obs.intersection(pos.index).intersection(ann.index)

if len(common) == 0:
    # ------------------------------------------------------------------
    # Soft fallback: proceed WITHOUT annotations/env_map if we can at least
    # align matrix barcodes with spatial positions.
    # ------------------------------------------------------------------
    common_pos = obs.intersection(pos.index)
    if len(common_pos) == 0:
        try:
            print(f"   - selected positions : {pos_path}")
        except Exception:
            pass
        try:
            print(f"   - selected annotation: {ann_path}")
        except Exception:
            pass
        print("   - matrix barcode sample    :", list(obs[:5]))
        print("   - positions barcode sample :", list(pos.index.astype(str)[:5]))
        print("   - annotation barcode sample:", list(ann.index.astype(str)[:5]))
        raise RuntimeError(
            "No overlapping barcodes between (h5 matrix) and (positions).\n"
            "This usually means you are using a .h5 that does NOT correspond to the binned Visium HD outputs.\n"
            "For Visium HD binned data, matrix barcodes and tissue_positions/annotations should share the SAME IDs.\n"
            "Action items:\n"
            "  - Double-check you are using the binned matrix (.h5) for the SAME bin size (16um) as positions/annotations.\n"
            "  - If your annotation barcodes look like 's_016um_00052_00082-1' but matrix barcodes look like 'AAAC...-1',\n"
            "    then this .h5 is a different namespace and cannot be joined.\n"
        )

    print(f"⚠️ [Preflight] Annotation barcodes do not match matrix/positions (common=0).")
    print(f"    Proceeding with positions-only common={len(common_pos)} and setting all labels to 'Unknown'.")
    common = common_pos
    labels = np.array(['Unknown'] * len(common), dtype=object)

    cfg.setdefault("data", {})["env_map"] = None
    cfg.setdefault("model", {})["irm_lambda"] = 0.0
    cfg["model"]["irm_balance"] = False

print(f"✅ [Preflight] Common barcodes: {len(common)} / matrix={len(obs)}")

coords = np.stack(
    [pos.loc[common, pos_x].to_numpy(), pos.loc[common, pos_y].to_numpy()],
    axis=1,
).astype(np.float32)
if 'labels' not in locals() or len(labels) != len(common):
    labels = ann.loc[common, ann_lab].astype(str).values

# ----------------------------------------------------------------------
# env_map 校验（增强鲁棒性）
# ----------------------------------------------------------------------
env_map = cast(dict, cfg.get("data", {})).get("env_map", None)
if isinstance(env_map, dict):
    labels_s = pd.Series(labels, dtype="string").astype(str).str.strip()
    labels_low = labels_s.str.lower()
    label_set = set(labels_low.unique().tolist())
    lower_to_orig = dict(zip(labels_low.tolist(), labels_s.tolist()))

    def _resolve_label_lower(target: str) -> Optional[str]:
        t = target.strip().lower()
        if t in label_set:
            return t
        synonyms = {
            "tumor": ["neoplasm", "tumour", "cancer", "carcinoma", "malignant"],
            "malignant": ["neoplasm", "tumor", "tumour", "cancer", "carcinoma"],
            "epithelial": ["epithelium", "glandular", "neoplasm"],
            "stroma": ["stromal", "smooth muscle", "muscle", "fibrous", "connective tissue"],
            "fibroblast": ["fibro", "stromal", "stroma"],
            "macrophage": ["macrophage", "mono", "myeloid"],
            "cd8+ t cell": ["cd8 t", "t cell", "t-cell", "cd8"],
            "unknown": ["unknown", "unassigned", "na", "n/a"],
            "necrosis": ["necrosis", "necrotic"],
            "artifact": ["artifact", "artefact"],
        }
        for syn in synonyms.get(t, []):
            if syn in label_set:
                return syn
        return None

    wanted = set()
    for k in ("env1_labels", "env2_labels", "ignore_labels"):
        vals = env_map.get(k, [])
        if isinstance(vals, list):
            wanted |= set(str(v).strip().lower() for v in vals)

    missing = wanted - label_set
    if not missing:
        print("✅ [Preflight] env_map labels consistent with annotation.")
    else:
        print("⚠️ [Preflight] env_map labels NOT found in annotation; auto-adapting env_map.")
        print(f"   - missing (sample): {sorted(list(missing))[:20]}")
        print(f"   - available labels (sample): {sorted(list(label_set))[:20]}")

        new_env_map_low: dict[str, list[str]] = {}
        for k in ("env1_labels", "env2_labels", "ignore_labels"):
            vals = env_map.get(k, [])
            resolved_low: list[str] = []
            if isinstance(vals, list):
                for v in vals:
                    r = _resolve_label_lower(str(v))
                    if r is not None and r not in resolved_low:
                        resolved_low.append(r)
            new_env_map_low[k] = resolved_low

        if len(new_env_map_low.get("env1_labels", [])) == 0:
            freq = labels_low.value_counts()
            tumor_like = [l for l in freq.index.tolist() if any(k in l for k in ("neo", "tum", "canc", "carcin", "malig"))]
            new_env_map_low["env1_labels"] = [tumor_like[0] if tumor_like else freq.index[0]]

        if len(new_env_map_low.get("env2_labels", [])) == 0:
            ignore = set(new_env_map_low.get("ignore_labels", []))
            env1 = set(new_env_map_low.get("env1_labels", []))
            others = [l for l in sorted(label_set) if l not in ignore and l not in env1]
            new_env_map_low["env2_labels"] = others[:5]

        def _to_orig(low_list: list[str]) -> list[str]:
            return [lower_to_orig.get(l, l) for l in low_list]

        cfg.setdefault("data", {})["env_map"] = {
            "env1_labels": _to_orig(new_env_map_low["env1_labels"]),
            "env2_labels": _to_orig(new_env_map_low["env2_labels"]),
            "ignore_labels": _to_orig(new_env_map_low.get("ignore_labels", [])),
        }
        print("✅ [Preflight] env_map adapted to your annotation labels:")
        print("   - env1_labels:", cfg["data"]["env_map"]["env1_labels"])
        print("   - env2_labels:", cfg["data"]["env_map"]["env2_labels"])
        print("   - ignore_labels:", cfg["data"]["env_map"]["ignore_labels"])

# ----------------------------------------------------------------------
# ✅ Pre-flight 阶段绝不做 HVG 截断（防 Marker 丢失）
# ----------------------------------------------------------------------
cfg.setdefault("data", {})
num_genes = int(cast(dict, cfg.get("data", {})).get("num_genes", 128))
cfg["data"]["num_genes"] = num_genes
print(f"✅ [Preflight] Keep full gene space here (num_genes target later={num_genes}).")

# Legacy compatibility only: keep X_csr available if a stale in-memory cell still references it.
if "X_csr" not in globals():
    try:
        X_csr = sp.csr_matrix(adata.X) if not sp.issparse(adata.X) else adata.X.tocsr()
    except Exception:
        X_csr = None

adata_common = adata[common].copy()
adata_common.obsm["spatial"] = coords
adata_common.obs["annotation"] = labels

adata = adata_common
print(f"✅ [Preflight] After barcode align: n_spots={adata.n_obs}, n_genes(full)={adata.n_vars}")


⚠️ [Preflight] No overlapping barcodes with current (pos_path, ann_path). Trying candidates auto-match...
⚠️ [Preflight] Annotation columns not named; using first two columns as (barcode, label).
⚠️ [Preflight] Annotation barcodes do not match matrix/positions (common=0).
    Proceeding with positions-only common=541968 and setting all labels to 'Unknown'.
✅ [Preflight] Common barcodes: 541968 / matrix=541968
✅ [Preflight] Keep full gene space here (num_genes target later=1024).
✅ [Preflight] After barcode align: n_spots=541968, n_genes(full)=18085


# ✅ Module 1 — ROI 截取 + 伪病理环境标签（Tumor/Stroma）生成（最紧急）

目标：
- 在 **HVG=128** 的表达矩阵上：PCA → Leiden
- 基于 marker score 自动（可半手动）把 cluster 映射成 `Tumor / Stroma / Unknown`，写入 `adata.obs['annotation']`
- 在全切片中自动截取 **Tumor–Stroma 交界 ROI**（保留 2000–3000 spots），并让后续 `annotation_csv.gz` 直接用 ROI 数据（防 OOM + 强化边界流场）


In [ ]:
# =========================
# Module 1 (Scanpy ROI + pseudo pathology labels)
# =========================
from __future__ import annotations

import os

import numpy as np
import pandas as pd
import scipy.sparse as sp

def _ensure_spatial_in_adata(adata):
    if "spatial" not in adata.obsm:
        for k in ["X_spatial", "spatial_hd", "spatial_coords"]:
            if k in adata.obsm:
                adata.obsm["spatial"] = np.asarray(adata.obsm[k])
                print(f"✅ [Module1] adata.obsm['spatial'] set from obsm['{k}']")
                break
    if "spatial" not in adata.obsm:
        raise KeyError("Missing adata.obsm['spatial'] (spatial coordinates).")


def _score_markers(adata, gene_list, score_name):
    import scanpy as sc

    # 统一大小写匹配（避免 var_names 大小写/前后空格导致 0 overlap）
    var = pd.Index(adata.var_names.astype(str)).str.strip()
    var_upper = pd.Index([v.upper() for v in var])
    want_upper = [g.upper() for g in gene_list]

    keep_mask = var_upper.isin(want_upper)
    keep = var[keep_mask].tolist()

    if len(keep) == 0:
        print(f"⚠️ [Module1] No marker genes found for {score_name}. Fill zeros and continue.")
        adata.obs[score_name] = 0.0
        return

    if len(keep) < 3:
        print(f"⚠️ [Module1] Marker overlap small for {score_name}: n={len(keep)} -> {keep}")

    # 若 adata.raw 存在（通常保存全基因），优先用 raw 打分以避免 HVG 截断导致 marker 丢失
    use_raw = getattr(adata, "raw", None) is not None
    sc.tl.score_genes(adata, gene_list=keep, score_name=score_name, use_raw=use_raw)

def run_leiden_and_pseudo_labels(
    adata,
    leiden_resolution: float = 0.6,
    random_state: int = 0,
    cluster_key: str = "leiden",
    out_key: str = "annotation",
    unknown_margin: float = 0.15,
    manual_cluster_map: dict[str, str] | None = None,
):
    '''
    Assumes adata already contains log1p normalized HVG matrix (n_genes ~ 128),
    and adata.obsm['spatial'] exists.

    Writes:
      - adata.obs[cluster_key] (Leiden)
      - adata.obs[out_key] in {'Tumor','Stroma','Unknown'}
    '''
    import scanpy as sc

    print("\n==============================")
    print("🚀 [Module1] PCA → Leiden → Pseudo labels (Tumor/Stroma)")
    print("==============================")
    _ensure_spatial_in_adata(adata)

    # 执行降维与聚类（不做 scale，避免稀疏矩阵密集化导致 OOM；若存在 highly_variable 列则仅用 HVG）
    if "highly_variable" in adata.var.columns:
        sc.tl.pca(adata, n_comps=min(50, adata.n_vars), use_highly_variable=True, svd_solver="arpack", random_state=random_state)
    else:
        sc.tl.pca(adata, n_comps=min(50, adata.n_vars), svd_solver="arpack", random_state=random_state)
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=min(30, adata.obsm["X_pca"].shape[1]), random_state=random_state)
    sc.tl.leiden(adata, resolution=leiden_resolution, random_state=random_state, key_added=cluster_key)

    tumor_markers = [
        "EPCAM","TACSTD2","KRT8","KRT18","KRT19","CEACAM5","CEACAM6","MUC1","MUC13","VIL1",
        "MKI67","TOP2A"
    ]
    stroma_markers = [
        "COL1A1","COL1A2","COL3A1","DCN","LUM","SPARC","TAGLN","ACTA2","VIM","PDGFRB","RGS5",
        "COL5A1","COL6A1","COL6A2"
    ]

    if "annotation" in adata.obs and "annotation_loaded" not in adata.obs:
        adata.obs["annotation_loaded"] = adata.obs["annotation"].astype(str).values


    # --- quick check: gene id system (symbol vs ENSG) ---
    vn = adata.var_names.astype(str)
    ens_like = float(np.mean([x.startswith("ENSG") for x in vn[: min(2000, len(vn))]])) if len(vn) else 0.0
    print(f"✅ [Module1] var_names example: {list(vn[:5])}")
    print(f"✅ [Module1] ENSG-like ratio (first 2k genes): {ens_like:.2f}")
    if ens_like > 0.5:
        print("⚠️ [Module1] var_names look like Ensembl IDs (ENSG...). Marker lists are SYMBOLS; overlap may be 0 unless you map IDs.")
    _score_markers(adata, tumor_markers, "_tumor_score")
    _score_markers(adata, stroma_markers, "_stroma_score")

    df = adata.obs[[cluster_key, "_tumor_score", "_stroma_score"]].copy()
    df["_delta"] = df["_tumor_score"] - df["_stroma_score"]
    g = df.groupby(cluster_key).agg(
        tumor_score=("_tumor_score","mean"),
        stroma_score=("_stroma_score","mean"),
        delta=("_delta","mean"),
        n=(cluster_key,"size"),
    ).sort_values("delta", ascending=False)

    delta = g["delta"].values
    scale = float(np.std(delta)) if len(delta) > 1 else 1.0
    thr = unknown_margin * max(scale, 1e-6)

    cluster2ann = {}
    for cl, row in g.iterrows():
        if row["delta"] > thr:
            cluster2ann[str(cl)] = "Tumor"
        elif row["delta"] < -thr:
            cluster2ann[str(cl)] = "Stroma"
        else:
            cluster2ann[str(cl)] = "Unknown"

    if manual_cluster_map:
        print("🛠️ [Module1] Applying manual_cluster_map:")
        for k, v in manual_cluster_map.items():
            print(f"  - {k} -> {v}")
            cluster2ann[str(k)] = str(v)

    disp = g.copy()
    disp["annotation"] = [cluster2ann[str(c)] for c in disp.index.astype(str)]
    print("\n[Module1] Cluster scoring summary (higher delta => Tumor-like):")
    print(disp.reset_index().to_string(index=False))

    adata.obs[out_key] = adata.obs[cluster_key].astype(str).map(cluster2ann).astype("category")
    print("\n✅ [Module1] annotation categories:", list(adata.obs[out_key].cat.categories))
    print("✅ [Module1] label counts:\n", adata.obs[out_key].astype(str).value_counts().to_string())

    return adata, cluster2ann


def _knn_graph(coords: np.ndarray, k: int = 6) -> sp.csr_matrix:
    from sklearn.neighbors import NearestNeighbors
    k = int(max(2, k))
    nbrs = NearestNeighbors(n_neighbors=min(k+1, coords.shape[0]), algorithm="auto").fit(coords)
    _, idx = nbrs.kneighbors(coords)
    rows = np.repeat(np.arange(coords.shape[0]), idx.shape[1]-1)
    cols = idx[:, 1:].reshape(-1)
    data = np.ones_like(rows, dtype=np.float32)
    A = sp.csr_matrix((data, (rows, cols)), shape=(coords.shape[0], coords.shape[0]))
    return A.maximum(A.T)


def crop_boundary_roi(
    adata,
    annotation_key: str = "annotation",
    target_min: int = 2000,
    target_max: int = 3000,
    spatial_k: int = 6,
    random_state: int = 0,
    n_centers: int = 5,
):
    """
    Boundary ROI (multi-cut / multi-truncation version):

      1) Build KNN graph on coords
      2) boundary nodes: have a neighbor of different label
      3) pick largest boundary connected component (interface backbone)
      4) choose multiple "cut points" (centers) along the backbone
      5) take union of per-center nearest neighborhoods, capped to [target_min, target_max]

    Why this helps:
      - A single Mahalanobis crop may over-focus on one boundary segment.
      - Multiple cut points cover a longer tumor–stroma interface, improving training signal diversity
        while keeping the total spot count bounded for CPU runs.
    """
    from scipy.sparse.csgraph import connected_components

    print("\n==============================")
    print("🎯 [Module1] Crop Tumor–Stroma boundary ROI (multi-center)")
    print("==============================")
    _ensure_spatial_in_adata(adata)

    coords = np.asarray(adata.obsm["spatial"], dtype=np.float32)
    ann = adata.obs[annotation_key].astype(str).values
    uniq = set(np.unique(ann))
    if len(uniq) < 2:
        print(f"⚠️ [Module1] annotation has <2 classes: {uniq}. ROI fallback to center crop.")
        center = np.median(coords, axis=0)
        d = np.linalg.norm(coords - center, axis=1)
        sel = np.argsort(d)[: min(target_max, adata.n_obs)]
        return adata[sel].copy(), sel

    A = _knn_graph(coords, k=spatial_k)

    codes = pd.Categorical(ann).codes
    r, c = A.nonzero()
    diff = codes[r] != codes[c]
    boundary = np.zeros(adata.n_obs, dtype=bool)
    if diff.any():
        boundary[r[diff]] = True
        boundary[c[diff]] = True
    else:
        print("⚠️ [Module1] No boundary edges detected; fallback to center crop.")
        center = np.median(coords, axis=0)
        d = np.linalg.norm(coords - center, axis=1)
        sel = np.argsort(d)[: min(target_max, adata.n_obs)]
        return adata[sel].copy(), sel

    b_idx = np.where(boundary)[0]
    Ab = A[b_idx][:, b_idx]
    n_comp, lab = connected_components(Ab, directed=False)
    sizes = np.bincount(lab) if len(lab) else np.array([0])
    best = int(np.argmax(sizes))
    keep_b = b_idx[lab == best]
    print(f"✅ [Module1] boundary nodes={len(b_idx)}; largest component size={len(keep_b)}")

    # Choose multiple cut points along the boundary backbone
    n_centers = int(max(1, min(n_centers, len(keep_b))))
    rng = np.random.default_rng(random_state)

    # Farthest-point sampling (robust, no sklearn dependency)
    backbone = coords[keep_b]
    centers = [int(rng.integers(0, backbone.shape[0]))]
    for _ in range(1, n_centers):
        chosen = backbone[centers]
        d2 = ((backbone[:, None, :] - chosen[None, :, :]) ** 2).sum(axis=2).min(axis=1)
        centers.append(int(np.argmax(d2)))
    center_xy = backbone[np.array(centers, dtype=int)]

    # Per-center neighborhood size (slightly oversample, then deduplicate + cap)
    per = int(np.ceil(target_max / float(n_centers)))
    sel_set: set[int] = set()
    for cx, cy in center_xy:
        d = np.sqrt(((coords[:, 0] - cx) ** 2) + ((coords[:, 1] - cy) ** 2))
        pick = np.argsort(d)[: min(per, coords.shape[0])]
        sel_set.update(map(int, pick))

    # Always include the boundary backbone itself
    sel_set.update(map(int, keep_b))
    sel = np.array(sorted(sel_set), dtype=int)

    # Enforce size bounds
    if sel.size < target_min:
        # add globally nearest-to-backbone mean as a fallback
        mu = backbone.mean(axis=0)
        d = np.linalg.norm(coords - mu, axis=1)
        more = np.argsort(d)[: min(target_min, coords.shape[0])]
        sel = np.unique(np.concatenate([sel, more]))

    if sel.size > target_max:
        sel = rng.choice(sel, size=target_max, replace=False)

    adata_roi = adata[sel].copy()
    print(f"✅ [Module1] ROI spots={adata_roi.n_obs} (target {target_min}-{target_max}, centers={n_centers})")
    print("✅ [Module1] ROI label counts:\n", adata_roi.obs[annotation_key].astype(str).value_counts().to_string())
    return adata_roi, sel



# -------------------------
# ✅ Run Module 1 here
# -------------------------
MANUAL_CLUSTER_MAP = None  # e.g. {"0":"Tumor","1":"Stroma"}

adata, _cluster2ann = run_leiden_and_pseudo_labels(
    adata,
    leiden_resolution=float(os.environ.get("LEIDEN_RES", "0.6")),
    random_state=int(os.environ.get("LEIDEN_SEED", "0")),
    manual_cluster_map=MANUAL_CLUSTER_MAP,
)

adata_roi, _roi_idx = crop_boundary_roi(
    adata,
    annotation_key="annotation",
    target_min=int(os.environ.get("ROI_MIN", "2000")),
    target_max=int(os.environ.get("ROI_MAX", "3000")),
    spatial_k=int(os.environ.get("ROI_K", "6")),
    random_state=int(os.environ.get("ROI_SEED", "0")),
    n_centers=int(os.environ.get("ROI_CENTERS", "5")),
)

# ✅ Switch downstream export to ROI
adata_train = adata_roi
common = pd.Index(adata_train.obs_names.astype(str))
coords = np.asarray(adata_train.obsm["spatial"], dtype=np.float32)
labels = adata_train.obs["annotation"].astype(str).values
gene_names = adata_train.var_names.astype(str).tolist()
X = adata_train.X


import numpy as np

def emergency_crop(X, coords, max_spots=5000): 
    """保命级裁剪：强制只取切片最中心的 max_spots 个点"""
    n_total = X.shape[0]
    if n_total <= max_spots:
        print(f"✅ 当前 Spot 数量 {n_total} 安全，无需裁剪。")
        return X, coords
    
    print(f"⚠️ 警告：检测到 {n_total} 个 Spot！正在强制裁剪至核心 {max_spots} 个...")
    # 找中心点
    center = coords.mean(axis=0)
    # 算所有点到中心的距离
    dists = np.linalg.norm(coords - center, axis=1)
    # 取距离最近的 max_spots 个点的索引
    keep_idx = np.argsort(dists)[:max_spots]
    
    # 切片
    X_cropped = X[keep_idx]
    coords_cropped = coords[keep_idx]
    print(f"✅ 裁剪完成！现在数据大小为: {X_cropped.shape}")
    
    return X_cropped, coords_cropped

# 👇 在加载完后，立刻加上这行！👇
# 为了保证后续 labels/common 与 (X, coords) 一致，这里在裁剪前先备份 coords，
# 然后用同一组 keep_idx 同步裁剪 labels 与 common（不改动其它流程）。
_coords_before_crop = np.asarray(coords, dtype=np.float32)
X, coords = emergency_crop(X, coords, max_spots=5000)

# ✅ 同步裁剪 labels 与 common，避免后续导出/训练出现长度不一致
if _coords_before_crop.shape[0] > 5000:
    _center = _coords_before_crop.mean(axis=0)
    _dists = np.linalg.norm(_coords_before_crop - _center, axis=1)
    _keep_idx = np.argsort(_dists)[:5000]
    labels = np.asarray(labels)[_keep_idx]
    common = pd.Index(np.asarray(common)[_keep_idx])


print("\n✅ [Module1] Training tensors switched to ROI:")
print("   - N_spots:", len(common))
print("   - N_genes:", len(gene_names))
print("   - coords:", coords.shape, "X:", (X.shape if hasattr(X, "shape") else type(X)))

# ✅ env_map auto-align (lowercase matching)
cfg.setdefault("data", {})
if cfg["data"].get("env_map", None) is None:
    cfg["data"]["env_map"] = {"env1_labels":["tumor"], "env2_labels":["stroma"], "ignore_labels":["unknown"]}
    print("✅ [Module1] cfg.data.env_map set to Tumor/Stroma/Unknown (lowercase match).")


🚀 [Module1] PCA → Leiden → Pseudo labels (Tumor/Stroma)


/home/liangrui/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: Please install the igraph package: `conda install -c conda-forge python-igraph` or `pip3 install igraph`.

In [ ]:
# ==============================================================================
# Pre-flight (2f): env_map 标签一致性检查 + 导出 annotation_csv.gz + 写 resolved config
# ==============================================================================
from __future__ import annotations

import csv, gzip
import numpy as np
import scipy.sparse as sp
from pathlib import Path
from typing import Optional
import os
import yaml

# env_map 标签检查（若 config 有 env_map）
env_map = cfg.get("data", {}).get("env_map", None)
if env_map is not None:
    present = set(map(lambda x: str(x).strip().lower(), np.unique(labels).tolist()))
    env1 = set(map(lambda x: str(x).strip().lower(), env_map.get("env1_labels", [])))
    env2 = set(map(lambda x: str(x).strip().lower(), env_map.get("env2_labels", [])))
    ign  = set(map(lambda x: str(x).strip().lower(), env_map.get("ignore_labels", [])))
    missing = (env1 | env2 | ign) - present
    if missing:
        print("⚠️ [Preflight] env_map labels NOT found in annotation; disabling env_map for this run.")
        print("   - missing (sample):", sorted(list(missing))[:50])
        print("   - available labels (sample):", sorted(list(present))[:50])
        cfg.setdefault('data', {})['env_map'] = None
    else:
        print("✅ [Preflight] env_map labels are consistent with annotations.")

OUT_DIR = PROJECT_ROOT / "runs" / "preprocessed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

annotation_csv = OUT_DIR / f"visium_hd_crc_{BIN_TAG}_{SAMPLE_TAG}_hvg{cfg['data']['num_genes']}.csv.gz"

# 确保数据包含基因表达列（从 X 和 gene_names）
chunk = 5000
N = len(common)

# 统一一个可迭代的条目顺序：common 的顺序
common_list = list(common)

# 获取对应 coords 与 labels 的顺序一致（我们前面就是按 common 生成）

# ✅ 修复：annotation_csv 只导出 (x,y,annotation)，不在 notebook 里写表达矩阵（避免提前截断/爆文件）
header = ["x", "y", "annotation"] + [str(g) for g in gene_names]  # genes added to header

# 使用 X 作为基因表达矩阵
X_expr = X  # 使用正确的基因表达矩阵 X（来自 adata_train.X）

with gzip.open(annotation_csv, "wt", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(header)

    # 确保写入基因表达数据（按 chunk 分块写入）
    is_sparse = sp.issparse(X_expr)
    for s in range(0, N, chunk):
        e = min(N, s + chunk)
        xb = coords[s:e, 0]
        yb = coords[s:e, 1]
        lb = labels[s:e]

        # 按稀疏/密集形式写入数据
        if is_sparse:
            block = X_expr[s:e, :].toarray()
        else:
            block = np.asarray(X_expr[s:e, :])

        # 确保是数值（train 端会检查 numeric）
        block = block.astype(np.float32)

        for i in range(e - s):
            w.writerow([float(xb[i]), float(yb[i]), str(lb[i])] + block[i, :].tolist())

print(f"✅ [Preflight] Wrote training-ready annotation_csv WITH expression: {annotation_csv}")

# ------------------------------------------------------------------------------
# N-control 防御（建议用于 Visium HD）：通过 grid_um 粗粒化降低 N，避免后续训练/显存爆炸
# 说明：visiumhd_crc_loader 会在读取 CSV 后执行 aggregate_to_grid(X, coords, grid_um=...).
# 如果你的 coords 以像素为单位（常见 pxl_*），grid_um 也应视为“像素尺度”的网格宽度。
# 这里基于坐标范围自动给出一个推荐 grid_um，使聚合后的网格 cell 数量约为 TARGET_GRID_CELLS。
# ------------------------------------------------------------------------------
TARGET_GRID_CELLS = int(os.environ.get("TARGET_GRID_CELLS", "20000"))  # 目标聚合后点数（建议 1e4~5e4）
if TARGET_GRID_CELLS > 0 and N > TARGET_GRID_CELLS:
    rx = float(coords[:, 0].max() - coords[:, 0].min())
    ry = float(coords[:, 1].max() - coords[:, 1].min())
    rec_grid = max(1.0, float((rx * ry / float(TARGET_GRID_CELLS)) ** 0.5))
    cur_grid = float(cfg.get("data", {}).get("grid_um", 0.0) or 0.0)
    if cur_grid <= 0:
        cfg["data"]["grid_um"] = rec_grid
        print(f"✅ [Preflight] Recommended grid_um={rec_grid:.2f} written to cfg (TARGET_GRID_CELLS={TARGET_GRID_CELLS}).")
    else:
        print(f"ℹ️ [Preflight] grid_um already set in cfg: {cur_grid} (recommended ~{rec_grid:.2f}).")
else:
    print(f"ℹ️ [Preflight] Skip N-control (N={N}, TARGET_GRID_CELLS={TARGET_GRID_CELLS}).")

# ------------------------------------------------------------------------------
# Hard-cap 防御：Visium HD 原始 N 往往 1e5~1e6；即使 grid 聚合到 2e4，ODE/GNN 仍可能爆显存。
# 这里把 “训练入口的最大点数” 作为第一优先级（<=5000 默认），否则 Windows 下容易 0xC0000005。
# 训练更稳的策略是：先在 ROI/子采样上把 pipeline 跑通，再逐步放大 max_cells。
# ------------------------------------------------------------------------------
MAX_CELLS_TRAIN = int(os.environ.get("MAX_CELLS_TRAIN", "1000"))
if MAX_CELLS_TRAIN > 0:
    cfg.setdefault("data", {})["max_cells"] = MAX_CELLS_TRAIN
    cfg["data"].setdefault("roi_mode", os.environ.get("ROI_MODE", "center"))
    cfg["data"].setdefault("roi_center_frac", float(os.environ.get("ROI_CENTER_FRAC", "0.4")))
    cfg["data"].setdefault("roi_seed", int(os.environ.get("ROI_SEED", str(cfg.get("seed", 0)))))
    print(f"✅ [Preflight] data.max_cells={cfg['data']['max_cells']} roi_mode={cfg['data']['roi_mode']}")

# ------------------------------------------------------------------------------
# Solver 防御：rk4 不接受 max_num_steps（已在 trainer 内做兼容）；但更推荐先用 dopri5+adjoint 跑通。
# ------------------------------------------------------------------------------
solver = cfg.setdefault("solver", {})

# 强制使用 rk4，而不进行修改
if str(solver.get("method", "")).lower() != "rk4":
    solver["method"] = "rk4"  # 设置为 rk4

# 设置 max_steps 为你希望的值，例如 5000
solver["max_steps"] = int(os.environ.get("SOLVER_MAX_STEPS", "100"))  # 修改为你需要的 max_steps

solver.setdefault("rtol", float(os.environ.get("SOLVER_RTOL", "1e-4")))
solver.setdefault("atol", float(os.environ.get("SOLVER_ATOL", "1e-5")))
solver.setdefault("adjoint_rtol", float(os.environ.get("SOLVER_ADJ_RTOL", str(solver["rtol"]))))  # For adjoint accuracy
solver.setdefault("adjoint_atol", float(os.environ.get("SOLVER_ADJ_ATOL", str(solver["atol"]))))
solver.setdefault("step_size", float(os.environ.get("SOLVER_STEP_SIZE", "0.1")))

print("✅ [Preflight] solver manually configured:", {k: solver.get(k) for k in ("method", "use_adjoint", "max_steps", "rtol", "atol", "step_size")})

# 写 resolved config：明确指向我们生成的 annotation_csv
cfg["data"]["name"] = "visium_hd_crc"
cfg["data"]["annotation_csv"] = str(annotation_csv.relative_to(PROJECT_ROOT))

# ------------------------------------------------------------------------------
# ✅ Run identity 防混淆：为真实数据运行设置唯一 save_dir / viz.out_dir，避免复用 demo 的 runs/model.pt
#   说明：Module 3 会从这个 save_dir 里找 pred_adj / checkpoints。
# ------------------------------------------------------------------------------
run_id = f"{BIN_TAG}_{SAMPLE_TAG}_hvg{cfg['data']['num_genes']}"
cfg.setdefault("train", {})["save_dir"] = f"runs/real_crc_{run_id}"
cfg.setdefault("viz", {})["out_dir"] = f"runs/real_crc_{run_id}/figures"

# ------------------------------------------------------------------------------
# ✅ 强一致性：训练 config 必须指向“gene symbol space 的 FULL gold GRN”（不做 HVG 截断）
#   你需要先运行 Module 2 生成 gold_grn_collectri_*_symbols.csv
# ------------------------------------------------------------------------------
expected_sym = OUT_DIR / f"gold_grn_collectri_{BIN_TAG}_{SAMPLE_TAG}_symbols.csv"
picked = None

if expected_sym.exists():
    picked = expected_sym
else:
    # best-effort: pick latest *_symbols.csv under OUT_DIR (in case BIN/SAMPLE tag changed)
    cand = sorted(OUT_DIR.glob("gold_grn_collectri_*_symbols.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if len(cand) > 0:
        picked = cand[0]

if picked is not None:
    cfg.setdefault("data", {})["gold_grn_path"] = str(picked.relative_to(PROJECT_ROOT))
    print(f"✅ [Preflight] data.gold_grn_path set (FULL symbol-space gold): {cfg['data']['gold_grn_path']}")
else:
    # If you already set a raw (gene-symbol) gold GRN in the base config, keep using it.
    if cfg.get("data", {}).get("gold_grn_path"):
        print("ℹ️ [Preflight] No *_symbols gold found in OUT_DIR; using base config data.gold_grn_path:", cfg["data"]["gold_grn_path"])
    else:
        print("⚠️ [Preflight] No gold GRN file found.")
        print("   Option A (recommended): run Module 2 to generate gold_grn_collectri_*_symbols.csv, then re-run this cell.")
        print("   Option B: manually set cfg['data']['gold_grn_path'] to a *gene-symbol* gold CSV (no HVG intersect).")

CFG_RESOLVED = OUT_DIR / "real_crc_config.resolved.yaml"
CFG_RESOLVED.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(f"✅ [Preflight] Resolved config written: {CFG_RESOLVED}")

# --- GRN export config (edge list) ---
# Default: jacobian (stable). If you *really* want IG, set method='ig_ok'.
cfg.setdefault('grn_export', {})
cfg['grn_export'].update({
    'enabled': True,
    'method': 'jacobian',
    'top_k': 2000,
    'abs_score': True,
    'ig_steps': 32,
})

# ---- enforce directed export (avoid jacobian_proxy/jacobian) ----
cfg.setdefault('grn_export', {})
cfg['grn_export']['method'] = 'cycle_jacobian'
print('[CFG] grn_export.method =', cfg['grn_export'].get('method'))


# ------------------------------------------------------------------------------
# Claim / pairing runtime controls
# ------------------------------------------------------------------------------
if "CLAIM_RUNTIME" not in globals():
    CLAIM_RUNTIME = {
        "claim_profile": "main_mechanism",
        "enable_ot_pairing": False,
        "enable_dynamic_anchor_pairing": False,
        "user_override_ot_pairing": None,
        "user_override_dynamic_anchor_pairing": None,
    }

cfg.setdefault("claim_runtime", {})
cfg["claim_runtime"].update(CLAIM_RUNTIME)

cfg.setdefault("model", {})
cfg["model"]["enable_ot"] = bool(CLAIM_RUNTIME.get("enable_ot_pairing", False))

cfg.setdefault("ot", {})
if not isinstance(cfg["ot"], dict):
    cfg["ot"] = {}
cfg["ot"]["enabled"] = bool(CLAIM_RUNTIME.get("enable_ot_pairing", False))

cfg.setdefault("data", {})
cfg["data"]["dynamic_anchor_pairing"] = bool(CLAIM_RUNTIME.get("enable_dynamic_anchor_pairing", False))

print("✅ [Preflight] Claim runtime applied:", {
    "claim_profile": CLAIM_RUNTIME.get("claim_profile"),
    "model.enable_ot": cfg["model"].get("enable_ot"),
    "ot.enabled": cfg["ot"].get("enabled"),
    "data.dynamic_anchor_pairing": cfg["data"].get("dynamic_anchor_pairing"),
})

# ------------------------------------------------------------------------------
# Write both resolved config and runtime config consumed by Train / Export / Module3
# ------------------------------------------------------------------------------
CFG_RUNTIME_PATH = PROJECT_ROOT / "runs" / "real_crc_config_nar_gpu_v2.yaml"
CFG_RUNTIME_PATH.parent.mkdir(parents=True, exist_ok=True)
CFG_RUNTIME_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(f"✅ [Preflight] Runtime config written: {CFG_RUNTIME_PATH}")


In [ ]:
# ==============================================================================
# Pre-flight (2g): 修复 kNN 构图 O(N^2) 爆内存（用 SciPy cKDTree 替代 torch.cdist）
#   背景：默认 knn_graph 使用 torch.cdist(coords, coords)，对 N~5e5 会尝试分配 ~TB 内存。
#   该 patch 会“就地修改” repo 内的 phycausal_stgrn/data/data_loader.py（自动备份一次），
#   使训练 CLI 子进程也能使用修复后的实现。
# ==============================================================================
from __future__ import annotations

import re, shutil
from pathlib import Path

PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", Path.cwd()))
dl_path = PROJECT_ROOT / "phycausal_stgrn" / "data" / "data_loader.py"
if not dl_path.exists():
    raise FileNotFoundError(f"Cannot find data_loader.py at: {dl_path}")

txt = dl_path.read_text(encoding="utf-8")

# --- Patch knn_graph ---
if "cKDTree" not in txt:
    backup = dl_path.with_suffix(".py.bak_before_ckdtree")
    if not backup.exists():
        shutil.copy2(dl_path, backup)
        print(f"✅ [Patch] Backup created: {backup.name}")

    knn_repl = """def knn_graph(coords: torch.Tensor, k: int) -> torch.Tensor:
    \"\"\"Build directed kNN graph (2, E) with a memory-safe backend.

    NOTE:
      The original implementation used `torch.cdist(coords, coords)` which allocates O(N^2) memory.
      For N~5e5 (Visium HD), that becomes ~TB and will crash with:
        DefaultCPUAllocator: not enough memory ... allocate 1e12 bytes

    This implementation prefers `scipy.spatial.cKDTree` (exact kNN, O(N log N) time, O(N) memory),
    and falls back to the original torch.cdist for small N if SciPy is unavailable.
    \"\"\"
    with torch.no_grad():
        coords_cpu = coords.detach().cpu().float()
        n = int(coords_cpu.shape[0])
        if n == 0:
            return torch.empty((2, 0), dtype=torch.long, device=coords.device)

        # Prefer SciPy KDTree (exact kNN, memory-safe)
        try:
            import numpy as _np
            from scipy.spatial import cKDTree  # type: ignore

            tree = cKDTree(_np.asarray(coords_cpu.numpy(), dtype=_np.float32))
            kk = int(k) + 1  # include self
            try:
                _, idx = tree.query(coords_cpu.numpy(), k=kk, workers=-1)
            except TypeError:
                # older SciPy without workers
                _, idx = tree.query(coords_cpu.numpy(), k=kk)

            knn = _np.asarray(idx[:, 1:], dtype=_np.int64)  # exclude self
            src = _np.repeat(_np.arange(n, dtype=_np.int64), int(k))
            dst = knn.reshape(-1)

            edge = _np.stack([src, dst], axis=0)
            edge_index = torch.from_numpy(edge).to(device=coords.device, dtype=torch.long)
            return edge_index

        except Exception:
            # Fallback (may OOM for huge N): only safe for small graphs
            d = torch.cdist(coords_cpu, coords_cpu)
            knn = torch.topk(d, k=int(k) + 1, largest=False).indices[:, 1:]
            src = torch.arange(n, device=coords_cpu.device).unsqueeze(1).repeat(1, int(k)).reshape(-1)
            dst = knn.reshape(-1)
            edge_index = torch.stack([src, dst], dim=0).to(device=coords.device, dtype=torch.long)
            return edge_index

"""

    pattern = r"def knn_graph\(coords: torch\.Tensor, k: int\) -> torch\.Tensor:[\s\S]*?return edge_index\s*\n\n"
    txt2, nsub = re.subn(pattern, knn_repl + "\n\n", txt, count=1)
    if nsub != 1:
        raise RuntimeError("Failed to patch knn_graph() in data_loader.py (pattern not found exactly once).")
    dl_path.write_text(txt2, encoding="utf-8")
    print("✅ [Patch] knn_graph() patched to SciPy cKDTree backend.")
else:
    print("✅ [Patch] knn_graph() already uses cKDTree (skipping).")


In [ ]:
# ==============================================================================
# Train: 使用 resolved config 开跑（带 cwd/env/编码防御 + 输出捕获）
# 并把本次运行的 CFG_PATH / TRAIN_SAVE_DIR 写回全局，供后续 cells 使用
# ==============================================================================
import os, sys, subprocess, gzip
from pathlib import Path
import yaml

CFG_PATH = Path(globals().get("CFG_RUNTIME_PATH", PROJECT_ROOT / "runs" / "real_crc_config_nar_gpu_v2.yaml"))
if not CFG_PATH.exists():
    raise FileNotFoundError(f"Config not found: {CFG_PATH}")

cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))

ann_csv = cfg.get("data", {}).get("annotation_csv", None)
if ann_csv:
    ann_path = (PROJECT_ROOT / ann_csv).resolve()
    if not ann_path.exists():
        raise FileNotFoundError(f"annotation_csv not found: {ann_path}")
    opener = gzip.open if str(ann_path).endswith(".gz") else open
    with opener(ann_path, "rt", encoding="utf-8", errors="replace") as f:
        head = [next(f) for _ in range(3)]
    print("✅ [Sanity] annotation_csv head:")
    for ln in head:
        print("   ", ln.rstrip()[:160])
else:
    print("⚠️ [Sanity] annotation_csv is not set in config (positions-only mode).")

solver = cfg.get("solver", {})
print(f"✅ [Sanity] solver.method={solver.get('method')} use_adjoint={solver.get('use_adjoint')}")

def _probe_memory():
    import shutil, subprocess
    print("\n=== [Probe] Host Memory ===")
    try:
        import psutil
        vm = psutil.virtual_memory()
        print(f"RAM total={vm.total/1e9:.2f}GB avail={vm.available/1e9:.2f}GB used={vm.percent}%")
    except Exception as e:
        print("psutil unavailable:", e)

    print("\n=== [Probe] CUDA Memory ===")
    try:
        import torch
        if torch.cuda.is_available() and str(cfg.get("device", "")).lower().startswith("cuda"):
            free, total = torch.cuda.mem_get_info()
            print(f"CUDA device={torch.cuda.get_device_name(0)} free={free/1e9:.2f}GB total={total/1e9:.2f}GB")
            print(f"torch.cuda.memory_reserved={torch.cuda.memory_reserved()/1e9:.2f}GB allocated={torch.cuda.memory_allocated()/1e9:.2f}GB")
        else:
            print("CUDA not active for this run (device=%s, cuda_available=%s)" % (cfg.get("device"), __import__("torch").cuda.is_available()))
    except Exception as e:
        print("torch cuda probe failed:", e)

    try:
        if shutil.which("nvidia-smi"):
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free", "--format=csv,noheader"],
                text=True,
            )
            print("\n=== [Probe] nvidia-smi ===")
            print(out.strip())
    except Exception as e:
        print("nvidia-smi probe failed:", e)

_probe_memory()

data_cfg = cfg.get("data", {})
if int(data_cfg.get("max_cells", 0) or 0) <= 0:
    print("⚠️ [Probe] data.max_cells is NOT set. For Visium HD, strongly recommend <=5000 for first successful run.")
else:
    print(f"✅ [Probe] data.max_cells={int(data_cfg.get('max_cells'))} roi_mode={data_cfg.get('roi_mode', 'none')}")

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONUTF8"] = "1"
env["PYTHONIOENCODING"] = "utf-8"
if str(cfg.get("device", "cpu")).lower() == "cpu":
    env["CUDA_VISIBLE_DEVICES"] = ""

cmd = [
    sys.executable, "-u",
    "-m", "phycausal_stgrn.cli",
    "train",
    "--config", str(CFG_PATH.resolve()),
    "--out_dir", "runs",
]
print("CWD:", str(PROJECT_ROOT))
print("RUN:", " ".join(cmd))

p = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)

assert p.stdout is not None
tail = []
TAIL_N = 200
for line in p.stdout:
    print(line, end="")
    tail.append(line)
    if len(tail) > TAIL_N:
        tail = tail[-TAIL_N:]

ret = p.wait()
print("\nreturncode:", ret)

if ret != 0:
    raise RuntimeError(
        "phycausal_stgrn.cli train failed with non-zero exit code.\n"
        f"Exit code: {ret}\n"
        "----- last logs (tail) -----\n"
        + "".join(tail)
    )

save_dir = (PROJECT_ROOT / str(cfg.get("train", {}).get("save_dir", "runs"))).resolve()

# 关键：回写全局，后续所有 cells 都优先用这两个
CFG_RUNTIME_PATH = CFG_PATH.resolve()
TRAIN_SAVE_DIR = save_dir
globals()["CFG_RUNTIME_PATH"] = CFG_RUNTIME_PATH
globals()["TRAIN_SAVE_DIR"] = TRAIN_SAVE_DIR

print("\n✅ [Post-run] CFG_RUNTIME_PATH =", CFG_RUNTIME_PATH)
print("✅ [Post-run] TRAIN_SAVE_DIR =", TRAIN_SAVE_DIR)

if save_dir.exists():
    entries = sorted(list(save_dir.glob("**/*")))[:200]
    print("✅ [Post-run] sample artifacts (up to 200 paths):")
    for pth in entries:
        if pth.is_file():
            print("   ", str(pth.relative_to(save_dir)))
else:
    print("⚠️ [Post-run] save_dir does not exist after training. Check trainer outputs / permissions.")


In [ ]:
# ==============================================================================
# Export-only: 从已有 checkpoint 直接导出 pred_edges（不再训练）
# 优先使用 manifest / stage1-best / gate-best，而不是只看 model.pt
# ==============================================================================
import os, sys, subprocess, json
from pathlib import Path
import yaml

CFG_PATH = Path(globals().get("CFG_RUNTIME_PATH", PROJECT_ROOT / "runs" / "real_crc_config_nar_gpu_v2.yaml")).resolve()
if not CFG_PATH.exists():
    raise FileNotFoundError(f"Config not found: {CFG_PATH}")

cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))
save_dir = Path(globals().get("TRAIN_SAVE_DIR", PROJECT_ROOT / str(cfg.get("train", {}).get("save_dir", "runs")))).resolve()

def _pick_ckpt(save_dir: Path) -> Path:
    manifest = save_dir / "training_manifest.json"
    if manifest.exists():
        try:
            m = json.loads(manifest.read_text(encoding="utf-8"))
            for key in ["preferred_eval_ckpt_realized", "preferred_eval_ckpt"]:
                v = m.get(key)
                if v:
                    p = (save_dir / v).resolve()
                    if p.exists():
                        return p
        except Exception:
            pass

    candidates = [
        save_dir / "model_stage1_best.pt",
        save_dir / "model_stage1_last.pt",
        save_dir / "model_best_gate.pt",
        save_dir / "model_best.pt",
        save_dir / "model.pt",
    ]
    epoch_candidates = sorted(save_dir.glob("model_epoch_*.pt"))
    candidates.extend(epoch_candidates)

    for p in candidates:
        if p.exists():
            return p.resolve()

    raise FileNotFoundError(f"No checkpoint found under: {save_dir}")

ckpt_path = _pick_ckpt(save_dir)
print("✅ [Export-only] using checkpoint:", ckpt_path)

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONUTF8"] = "1"
env["PYTHONIOENCODING"] = "utf-8"
if str(cfg.get("device", "cpu")).lower() == "cpu":
    env["CUDA_VISIBLE_DEVICES"] = ""

cmd = [
    sys.executable, "-u",
    "-m", "phycausal_stgrn.cli", "export-edges",
    "--config", str(CFG_PATH),
    "--ckpt", str(ckpt_path),
    "--out_dir", str(save_dir),
]
print(" ".join(cmd))
ret = subprocess.call(cmd, cwd=str(PROJECT_ROOT), env=env)
print("returncode:", ret)
if ret != 0:
    raise RuntimeError(f"export-edges failed with exit code={ret}")


In [ ]:
# --- Post-run: check main-claim outputs (prefer runtime save_dir, never old hardcoded dir first) ---
from pathlib import Path
import json
import yaml

PROJECT_ROOT = Path("/home/liangrui/local_mount/PhyCausal-STGRN-demo-plus")

CFG_PATH = Path(globals().get("CFG_RUNTIME_PATH", PROJECT_ROOT / "runs" / "real_crc_config_nar_gpu_v2.yaml")).resolve()
cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8")) if CFG_PATH.exists() else {}

CONFIG_SAVE_DIR = (PROJECT_ROOT / str(cfg.get("train", {}).get("save_dir", "runs"))).resolve()
DEFAULT_SAVE_DIR = CONFIG_SAVE_DIR

cands = []
for x in [
    globals().get("TRAIN_SAVE_DIR", None),
    globals().get("train_save_dir", None),
    CONFIG_SAVE_DIR,
    DEFAULT_SAVE_DIR,
]:
    if x is None:
        continue
    try:
        p = Path(x).resolve()
        if p not in cands:
            cands.append(p)
    except Exception:
        pass

print("=== Checked save dirs ===")
for d in cands:
    print("-", d)

main_outputs = [
    "model.pt",
    "model_best.pt",
    "model_stage1_best.pt",
    "training_manifest.json",
    "pred_adj.npy",
    "pred_adj_raw.npy",
    "paper_summary.json",
    "counterfactual_summary.json",
    "gate_summary.json",
]

edge_outputs = [
    "pred_edges.pt",
    "pred_edges.csv",
    "pred_edges_raw.pt",
    "pred_edges_raw.csv",
    "pred_edges_error.txt",
]

aux_outputs = [
    "genes_128.txt",
    "genes_256.txt",
    "genes_512.txt",
    "genes_1024.txt",
]

found_main = False
found_edge = False
found_gene = False

for d in cands:
    if not d.exists():
        print(f"\n❌ save_dir not found: {d}")
        continue

    print(f"\n=== Inspecting: {d} ===")

    for fname in main_outputs:
        p = d / fname
        ok = p.exists()
        print(("✅" if ok else "❌"), p)
        if ok:
            found_main = True

    for fname in edge_outputs:
        p = d / fname
        ok = p.exists()
        print(("✅" if ok else "❌"), p)
        if ok and fname.endswith((".pt", ".csv")):
            found_edge = True

    for fname in aux_outputs:
        p = d / fname
        ok = p.exists()
        print(("✅" if ok else "❌"), p)
        if ok:
            found_gene = True

if not found_gene:
    raise RuntimeError("❌ 未找到 genes_*.txt。请先确认训练是否完整结束，或 TRAIN_SAVE_DIR / config.save_dir 是否正确。")

if not found_main and not found_edge:
    raise RuntimeError("❌ 未找到主线输出也未找到 edge 输出。请先检查训练是否完成，以及 save_dir 是否正确。")

print("\n🎯 Post-run check finished.")


# ✅ Module 2 — Gold Standard GRN 接入（OmniPath / CollecTRI）

- 自动拉取 CollecTRI TF→Target 网络（TSV）
- 与当前 HVG(128) 基因集合取交集
- 生成 `gold_grn.csv`，严格三列：`src_gene,dst_gene,label`


In [ ]:
# =========================
# Module 2 (Gold Standard GRN: optional rebuild, main-claim safe)
# 目标：
#   - 有 EXT + idmapping 时：显性重建 gold
#   - 缺任一文件时：不报错，直接跳过 gold，主线继续
# 输出：
#   gold_grn_path = Path | None
#   gold_df = DataFrame | None
# =========================
from __future__ import annotations

from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path("/home/liangrui/local_mount/PhyCausal-STGRN-demo-plus")
OUT_DIR = PROJECT_ROOT / "runs" / "preprocessed"
CACHE_DIR = OUT_DIR / "_cache"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if "BIN_TAG" not in globals() or "SAMPLE_TAG" not in globals():
    raise NameError("请先确保 BIN_TAG / SAMPLE_TAG 已在 notebook 上游定义。")
if "gene_names" not in globals():
    raise NameError("请先确保 gene_names 已在 notebook 上游定义。")

_RE6_1 = re.compile(r'\b[OPQ][0-9][A-Z0-9]{3}[0-9]\b')
_RE6_2 = re.compile(r'\b[A-NR-Z][0-9]{5}\b')
_RE10  = re.compile(r'\b[A-NR-Z][0-9][A-Z0-9]{8}\b')

def _extract_uniprot_accessions(s: str) -> list[str]:
    if s is None:
        return []
    s = str(s).strip()
    accs = set()
    for rgx in (_RE10, _RE6_1, _RE6_2):
        accs.update(rgx.findall(s))
    return sorted(accs)

def _build_uniprot_to_symbol_dict(idmap_path: Path) -> dict[str, str]:
    if idmap_path.suffix == ".gz":
        df = pd.read_csv(idmap_path, sep="\t", compression="gzip")
    else:
        df = pd.read_csv(idmap_path, sep="\t")
    need = {"From", "Gene Names"}
    if not need.issubset(df.columns):
        raise ValueError(f"❌ Mapping file must contain columns {need}, got {list(df.columns)}")
    df["From"] = df["From"].astype(str).str.strip()
    df["Gene Names"] = df["Gene Names"].astype(str).str.strip()
    df["symbol"] = df["Gene Names"].str.split().str[0].fillna("").astype(str).str.strip().str.upper()
    df = df[(df["From"] != "") & (df["symbol"] != "")]
    m = {}
    for a, sym in zip(df["From"], df["symbol"]):
        if a not in m:
            m[a] = sym
    return m

def _map_field_to_symbols(field: str, uniprot2sym: dict[str, str]) -> list[str]:
    accs = _extract_uniprot_accessions(field)
    out, seen = [], set()
    for a in accs:
        sym = uniprot2sym.get(a, "")
        if sym and sym not in seen:
            seen.add(sym)
            out.append(sym)
    return out

def build_gold_grn_from_ext(ext_csv: Path, out_csv: Path, idmap_path: Path, gene_names: list[str], expand_complex: bool = True, min_edges_for_hvg: int = 100):
    uniprot2sym = _build_uniprot_to_symbol_dict(idmap_path)
    print(f"✅ [Module2] Loaded UniProt->Symbol map: {len(uniprot2sym):,}")
    raw = pd.read_csv(ext_csv)
    if not {"src_gene", "dst_gene"}.issubset(raw.columns):
        raise ValueError(f"❌ EXT CSV must have columns src_gene/dst_gene, got {list(raw.columns)}")
    edges = []
    for src, dst in zip(raw["src_gene"], raw["dst_gene"]):
        src_syms = _map_field_to_symbols(src, uniprot2sym)
        dst_syms = _map_field_to_symbols(dst, uniprot2sym)
        if not src_syms or not dst_syms:
            continue
        if expand_complex:
            for s in src_syms:
                for t in dst_syms:
                    if s != t:
                        edges.append((s, t, 1))
        else:
            if src_syms[0] != dst_syms[0]:
                edges.append((src_syms[0], dst_syms[0], 1))
    df = pd.DataFrame(edges, columns=["src_gene", "dst_gene", "label"]).drop_duplicates(["src_gene", "dst_gene"])
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    sym_csv = out_csv.with_name(out_csv.name.replace(f"_hvg{len(gene_names)}", "_symbols"))
    df.to_csv(sym_csv, index=False)
    print(f"✅ [Module2] Saved full symbol-space gold: {sym_csv}")
    hvg = set(pd.Series(gene_names).astype(str).str.upper().tolist())
    df_hvg = df[df["src_gene"].isin(hvg) & df["dst_gene"].isin(hvg)].copy()
    df_hvg.to_csv(out_csv, index=False)
    print(f"✅ [Module2] Saved HVG-intersected gold: {out_csv}")
    print(f"✅ [Module2] full edges={len(df):,} | hvg edges={len(df_hvg):,} | hvg genes={len(hvg):,}")
    if len(df_hvg) < min_edges_for_hvg:
        print(f"⚠️ [Module2] HVG gold too small (<{min_edges_for_hvg}), fallback to symbol-space gold.")
        return sym_csv, df
    return out_csv, df_hvg

ext_candidates = [
    OUT_DIR / f"gold_grn_collectri_{BIN_TAG}_{SAMPLE_TAG}_EXT.csv",
    OUT_DIR / f"gold_grn_collectri_{BIN_TAG}_{SAMPLE_TAG}_ext.csv",
]
EXT_PATH = next((p for p in ext_candidates if p.exists()), None)
idmap_candidates = sorted(CACHE_DIR.glob("idmapping*.tsv")) + sorted(CACHE_DIR.glob("idmapping*.tsv.gz"))
IDMAP_PATH = idmap_candidates[0] if idmap_candidates else None

gold_grn_path = None
gold_df = None

if EXT_PATH is None:
    print("⚠️ [Module2] 未找到 *_EXT.csv。当前主线不依赖 gold，跳过 gold 重建。")
elif IDMAP_PATH is None:
    print("⚠️ [Module2] 未找到 idmapping*.tsv / .tsv.gz。当前主线不依赖 gold，跳过 gold 重建。")
else:
    try:
        OUT_GOLD = OUT_DIR / f"gold_grn_collectri_{BIN_TAG}_{SAMPLE_TAG}_hvg{len(gene_names)}.csv"
        gold_grn_path, gold_df = build_gold_grn_from_ext(EXT_PATH, OUT_GOLD, IDMAP_PATH, gene_names, expand_complex=True)
        print(f"✅ [Module2] gold_grn_path = {gold_grn_path}")
    except Exception as e:
        print(f"⚠️ [Module2] gold 重建失败，但不影响主线继续：{e}")
        gold_grn_path = None
        gold_df = None

globals()["gold_grn_path"] = gold_grn_path
globals()["gold_df"] = gold_df

print("\n=== Module2 summary ===")
print("gold_grn_path =", gold_grn_path)
print("gold_enabled =", gold_grn_path is not None)


# ✅ Module 3 — 定量评估 + 快照保存 + 自动调用 scripts/paper_figures.py

- 从训练产物中导出推断 GRN 邻接矩阵（best-effort 自动探测）
- 保存推断快照：genes / coords / pred_adj / gold_grn_path
- 自动调用 `scripts/paper_figures.py`（会先解析 `--help` 推断参数；失败则尝试多种参数组合）
- 若脚本失败：Notebook 内部计算 **AUPR + PR 曲线** 并落盘（保证你有定量结果）


In [ ]:
# =========================
# Module 3 (run-isolated eval + task-aligned eval + figures)
# 主线优先版：强制优先使用 manifest / stage1-best ckpt，避免拿错 model.pt
# =========================
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path("/home/liangrui/local_mount/PhyCausal-STGRN-demo-plus")

def _now_tag() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def _safe_float(x):
    try:
        if x is None:
            return None
        return float(x)
    except Exception:
        return None

def _copy_if_exists(src: Path | None, out_dir: Path):
    if src is not None and src.exists() and src.is_file():
        shutil.copy2(src, out_dir / src.name)

def _load_json(path: Path) -> dict | None:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def _find_in_save_dir(save_dir: Path, names: list[str]) -> Path | None:
    for name in names:
        p = save_dir / name
        if p.exists() and p.is_file():
            return p
    return None

def _resolve_eval_ckpt(save_dir: Path) -> Path | None:
    manifest_path = save_dir / "training_manifest.json"
    if manifest_path.exists():
        m = _load_json(manifest_path) or {}
        for key in ["preferred_eval_ckpt_realized", "preferred_eval_ckpt"]:
            v = m.get(key)
            if v:
                p = (save_dir / v).resolve()
                if p.exists() and p.is_file():
                    return p

    ordered = [
        "model_stage1_best.pt",
        "model_stage1_last.pt",
        "model_best_gate.pt",
        "model_best.pt",
        "model.pt",
        "checkpoint.pt",
    ]
    for name in ordered:
        p = save_dir / name
        if p.exists() and p.is_file():
            return p.resolve()

    epoch_ckpts = sorted(save_dir.glob("model_epoch_*.pt"))
    if epoch_ckpts:
        return epoch_ckpts[-1].resolve()
    return None

def _read_pred_edges_csv(csv_path: Path, genes_upper: list[str]):
    import csv
    idx = {g.upper(): i for i, g in enumerate(genes_upper)}
    edge_i, edge_j, scores = [], [], []
    with csv_path.open("r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            src = str(row.get("src") or row.get("source") or row.get("tf") or "").strip().upper()
            dst = str(row.get("dst") or row.get("target") or row.get("gene") or "").strip().upper()
            sc = row.get("score") or row.get("weight") or row.get("prob") or row.get("edge_score")
            if src in idx and dst in idx and sc is not None:
                try:
                    edge_i.append(idx[src]); edge_j.append(idx[dst]); scores.append(float(sc))
                except Exception:
                    pass
    if not scores:
        raise RuntimeError(f"No valid edges parsed from {csv_path}")
    return np.vstack([np.asarray(edge_i), np.asarray(edge_j)]), np.asarray(scores, dtype=np.float32)

def _load_pred_edges_pt(pt_path: Path):
    import torch
    obj = torch.load(pt_path, map_location="cpu")
    if isinstance(obj, dict):
        edge_index = obj.get("edge_index")
        scores = obj.get("scores") or obj.get("edge_score") or obj.get("weights")
        edge_gene_names = obj.get("edge_gene_names")
    else:
        raise RuntimeError(f"Unsupported pred_edges.pt format: {type(obj)}")
    if edge_index is None or scores is None:
        raise RuntimeError("pred_edges.pt missing edge_index or scores")
    edge_index = np.asarray(edge_index, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float32)
    return edge_index, scores, edge_gene_names

def edges_to_dense(edge_index, scores, G):
    adj = np.zeros((G, G), dtype=np.float32)
    if edge_index.shape[0] != 2:
        raise RuntimeError(f"edge_index shape must be [2, E], got {edge_index.shape}")
    adj[edge_index[0], edge_index[1]] = scores
    np.fill_diagonal(adj, 0.0)
    return adj

def save_snapshot(out_dir: Path, genes, coords, pred_adj, gold_grn_path=None):
    out_dir.mkdir(parents=True, exist_ok=True)
    npz_path = out_dir / f"snapshot_{_now_tag()}.npz"
    np.savez_compressed(
        npz_path,
        genes=np.asarray(genes, dtype=object),
        coords=np.asarray(coords, dtype=np.float32),
        pred_adj=np.asarray(pred_adj, dtype=np.float32),
        gold_grn_path=str(gold_grn_path) if gold_grn_path else "",
    )
    return npz_path

CFG_PATH = Path(globals().get("CFG_RUNTIME_PATH", PROJECT_ROOT / "runs" / "real_crc_config_nar_gpu_v2.yaml")).resolve()
if not CFG_PATH.exists():
    raise FileNotFoundError(f"Config not found: {CFG_PATH}")

cfg3 = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))
save_dir = Path(globals().get("TRAIN_SAVE_DIR", PROJECT_ROOT / str(cfg3.get("train", {}).get("save_dir", "runs")))).resolve()
if not save_dir.exists():
    raise FileNotFoundError(f"save_dir not found: {save_dir}")

TRAIN_SAVE_DIR = save_dir
globals()["TRAIN_SAVE_DIR"] = TRAIN_SAVE_DIR

bin_tag = globals().get("BIN_TAG", os.environ.get("PHYCAUSAL_BIN_TAG", "8um"))
OUT_EVAL = PROJECT_ROOT / "runs" / "eval" / bin_tag / f"{save_dir.name}__{_now_tag()}"
OUT_EVAL.mkdir(parents=True, exist_ok=True)

print("✅ [Module3] train.save_dir =", save_dir)
print("✅ [Module3] out_dir =", OUT_EVAL)

genes_file = _find_in_save_dir(save_dir, ["genes_512.txt", "genes_1024.txt", "genes_256.txt", "genes_128.txt"])
if genes_file is None:
    raise FileNotFoundError(f"No genes_*.txt found under: {save_dir}")

genes = [x.strip() for x in genes_file.read_text(encoding="utf-8", errors="replace").splitlines() if x.strip()]
genes_upper = [g.upper() for g in genes]
G = len(genes)

gold_csv_candidates = [
    PROJECT_ROOT / "runs" / "preprocessed" / f"gold_grn_collectri_{bin_tag}_GSM8594569_P5CRC_symbols.csv",
    PROJECT_ROOT / "runs" / "preprocessed" / "gold_grn_collectri_8um_GSM8594569_P5CRC_symbols.csv",
]
gold_csv_path = next((p for p in gold_csv_candidates if p.exists()), None)
if gold_csv_path is None:
    print("⚠️ [Module3] 未找到 gold GRN CSV。将跳过 edge/dense gold eval，仅保留主任务评估。")

pred_adj_cal_path = _find_in_save_dir(save_dir, ["pred_adj.npy"])
pred_adj_raw_path = _find_in_save_dir(save_dir, ["pred_adj_raw.npy"])
pred_edges_pt_path = _find_in_save_dir(save_dir, ["pred_edges.pt", "pred_edges_raw.pt"])
pred_edges_csv_path = _find_in_save_dir(save_dir, ["pred_edges.csv", "pred_edges_raw.csv"])
paper_summary_path = _find_in_save_dir(save_dir, ["paper_summary.json"])
counterfactual_summary_path = _find_in_save_dir(save_dir, ["counterfactual_summary.json"])
gate_summary_path = _find_in_save_dir(save_dir, ["gate_summary.json"])

manifest_path = save_dir / "training_manifest.json"
manifest = _load_json(manifest_path) if manifest_path.exists() else None
ckpt_path = _resolve_eval_ckpt(save_dir)
if ckpt_path is None:
    raise RuntimeError(f"❌ [Module3] Cannot resolve eval checkpoint under: {save_dir}")

for p in [pred_adj_cal_path, pred_adj_raw_path, pred_edges_pt_path, pred_edges_csv_path, paper_summary_path, counterfactual_summary_path, gate_summary_path, ckpt_path, genes_file, gold_csv_path, manifest_path]:
    _copy_if_exists(p, OUT_EVAL)

pred_adj_cal = None
pred_adj_raw = None
if pred_adj_cal_path is not None:
    pred_adj_cal = np.asarray(np.load(pred_adj_cal_path), dtype=np.float32)
    np.fill_diagonal(pred_adj_cal, 0.0)
    if pred_adj_cal.shape != (G, G):
        print(f"⚠️ [Module3] pred_adj.npy shape mismatch: expect {(G, G)}, got {pred_adj_cal.shape}")
        pred_adj_cal = None

if pred_adj_raw_path is not None:
    pred_adj_raw = np.asarray(np.load(pred_adj_raw_path), dtype=np.float32)
    np.fill_diagonal(pred_adj_raw, 0.0)
    if pred_adj_raw.shape != (G, G):
        print(f"⚠️ [Module3] pred_adj_raw.npy shape mismatch: expect {(G, G)}, got {pred_adj_raw.shape}")
        pred_adj_raw = None

if pred_adj_cal is None and pred_edges_pt_path is not None:
    try:
        edge_index, scores, edge_gene_names = _load_pred_edges_pt(pred_edges_pt_path)
        pred_adj_cal = edges_to_dense(edge_index=edge_index, scores=scores, G=G)
        np.save(OUT_EVAL / "pred_adj_from_edges.npy", pred_adj_cal.astype(np.float32))
        print("✅ [Module3] built pred_adj_from_edges.npy")
    except Exception as e:
        print(f"⚠️ [Module3] failed to load pred_edges.pt: {e}")

if pred_adj_cal is None and pred_edges_csv_path is not None:
    try:
        edge_index, scores = _read_pred_edges_csv(pred_edges_csv_path, genes_upper=genes_upper)
        pred_adj_cal = edges_to_dense(edge_index=edge_index, scores=scores, G=G)
        np.save(OUT_EVAL / "pred_adj_from_csv.npy", pred_adj_cal.astype(np.float32))
        print("✅ [Module3] built pred_adj_from_csv.npy")
    except Exception as e:
        print(f"⚠️ [Module3] failed to load pred_edges.csv: {e}")

if pred_adj_cal is None and pred_adj_raw is None:
    raise RuntimeError("❌ [Module3] Could not locate any predicted adjacency or edge list under current save_dir.")

pred_adj_for_snapshot = pred_adj_cal if pred_adj_cal is not None else pred_adj_raw

if "coords" not in globals():
    raise NameError("`coords` is not defined in notebook scope.")
coords_np = np.asarray(coords, dtype=np.float32)
snapshot_npz = save_snapshot(OUT_EVAL, genes=genes, coords=coords_np, pred_adj=pred_adj_for_snapshot, gold_grn_path=gold_csv_path)

paper_summary = _load_json(paper_summary_path) if paper_summary_path else None
counterfactual_summary = _load_json(counterfactual_summary_path) if counterfactual_summary_path else None
gate_summary = _load_json(gate_summary_path) if gate_summary_path else None
paper_fig_rc = None

paper_fig_candidates = [
    PROJECT_ROOT / "phycausal_stgrn" / "scripts" / "paper_figures.py",
    PROJECT_ROOT / "scripts" / "paper_figures.py",
]
paper_fig = next((p for p in paper_fig_candidates if p.exists() and p.is_file()), None)
if paper_fig is not None and ckpt_path is not None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTHONUTF8"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    fig_out_dir = OUT_EVAL / "paper_figures"
    cmd = [
        sys.executable, str(paper_fig),
        "--config", str(CFG_PATH),
        "--ckpt", str(ckpt_path),
        "--out_dir", str(fig_out_dir.resolve()),
        "--edge_method", "jacobian",
        "--top_k", "2000",
    ]
    print("🧪 [Module3] Running:", " ".join(cmd))
    paper_fig_rc = subprocess.call(cmd, cwd=str(PROJECT_ROOT), env=env)
    if paper_fig_rc == 0:
        paper_summary_path = fig_out_dir / "paper_summary.json"
        counterfactual_summary_path = fig_out_dir / "counterfactual_summary.json"
        gate_summary_path = fig_out_dir / "gate_summary.json"
        paper_summary = _load_json(paper_summary_path)
        counterfactual_summary = _load_json(counterfactual_summary_path)
        gate_summary = _load_json(gate_summary_path)

def _pick(d: dict | None, *keys):
    if not d:
        return None
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None

p_cross_baseline = _safe_float(_pick(paper_summary, "p_cross_baseline"))
p_cross_do_z = _safe_float(_pick(paper_summary, "p_cross_do_z"))
if p_cross_baseline is None:
    p_cross_baseline = _safe_float(_pick(counterfactual_summary, "p_cross_baseline"))
if p_cross_do_z is None:
    p_cross_do_z = _safe_float(_pick(counterfactual_summary, "p_cross_do_z"))

delta_p_cross = None
if p_cross_baseline is not None and p_cross_do_z is not None:
    delta_p_cross = p_cross_do_z - p_cross_baseline

_claim_runtime = {
    "claim_profile": "main_mechanism",
    "enable_ot_pairing": bool(cfg3.get("model", {}).get("enable_ot_pairing", False)),
    "enable_dynamic_anchor_pairing": bool(cfg3.get("model", {}).get("enable_dynamic_anchor_pairing", False)),
    "user_override_ot_pairing": None,
    "user_override_dynamic_anchor_pairing": None,
}

task_aligned = {
    "claim_runtime": _claim_runtime,
    "task_primary": {
        "delta_p_cross": delta_p_cross,
        "p_cross_baseline": p_cross_baseline,
        "p_cross_do_z": p_cross_do_z,
        "p_cross_envelope_do_z": _safe_float(_pick(paper_summary, "p_cross_envelope_do_z")),
        "gate_mean_interface": _safe_float(_pick(paper_summary, "gate_mean_interface", "gate_interface_mean")),
        "gate_mean_non_interface": _safe_float(_pick(paper_summary, "gate_mean_non_interface", "gate_non_interface_mean")),
        "gate_sparsity": _safe_float(_pick(paper_summary, "gate_sparsity")),
        "ood_flag_do_z": _safe_float(_pick(paper_summary, "ood_flag_do_z")),
    },
    "task_secondary": {
        "gold_enabled": gold_csv_path is not None,
        "edge_eval_status": "gold_found" if gold_csv_path is not None else "skipped_no_gold",
    },
    "artifacts": {
        "config_path": str(CFG_PATH),
        "save_dir": str(save_dir),
        "snapshot_npz": str(snapshot_npz),
        "pred_adj_calibrated": str(pred_adj_cal_path) if pred_adj_cal_path else None,
        "pred_adj_raw": str(pred_adj_raw_path) if pred_adj_raw_path else None,
        "pred_edges_pt": str(pred_edges_pt_path) if pred_edges_pt_path else None,
        "pred_edges_csv": str(pred_edges_csv_path) if pred_edges_csv_path else None,
        "checkpoint": str(ckpt_path) if ckpt_path else None,
        "ckpt_used_name": ckpt_path.name if ckpt_path else None,
        "preferred_eval_ckpt": manifest.get("preferred_eval_ckpt") if manifest else None,
        "preferred_eval_ckpt_realized": manifest.get("preferred_eval_ckpt_realized") if manifest else None,
        "gold_csv": str(gold_csv_path) if gold_csv_path else None,
        "paper_summary_json": str(paper_summary_path) if paper_summary_path else None,
        "counterfactual_summary_json": str(counterfactual_summary_path) if counterfactual_summary_path else None,
        "gate_summary_json": str(gate_summary_path) if gate_summary_path else None,
        "paper_figures_returncode": paper_fig_rc,
    },
    "notes": {
        "task_primary_interpretation": "主任务是空间向量场 / 反事实 rollout / Gate / Tumor-Stroma 屏障 / OOD dual rollout；gold edge eval 为可选辅助。",
        "gold_missing_behavior": "No gold -> skip edge eval, continue main-claim evaluation." if gold_csv_path is None else "Gold found -> edge eval available.",
        "ot_pairing_effective_in_backend": bool(cfg3.get("model", {}).get("enable_ot", False)),
        "dynamic_anchor_pairing_notebook_only": True,
    },
}

(OUT_EVAL / "task_aligned_evaluation.json").write_text(json.dumps(task_aligned, indent=2, ensure_ascii=False), encoding="utf-8")
summary = {
    "claim_runtime": _claim_runtime,
    "task_aligned_evaluation": task_aligned,
    "paper_summary": paper_summary,
    "counterfactual_summary": counterfactual_summary,
    "gate_summary": gate_summary,
    "manifest": manifest,
}
(OUT_EVAL / "evaluation_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("✅ [Module3] DONE. Outputs:", OUT_EVAL)
print(json.dumps(task_aligned, indent=2, ensure_ascii=False))


In [ ]:
# =========================
# Module 3.5 helper (rebuild main-claim summaries and copy back)
# 修正：优先使用 CFG_RUNTIME_PATH 和 TRAIN_SAVE_DIR，避免旧默认路径覆盖新实验
# =========================
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/liangrui/local_mount/PhyCausal-STGRN-demo-plus")

CFG_PATH = Path(globals().get("CFG_RUNTIME_PATH", PROJECT_ROOT / "runs" / "real_crc_config_nar_gpu_v2.yaml")).resolve()
if not CFG_PATH.exists():
    raise FileNotFoundError(f"Missing config: {CFG_PATH}")

save_dir = Path(globals().get("TRAIN_SAVE_DIR", PROJECT_ROOT / "runs")).resolve()
if not save_dir.exists():
    raise FileNotFoundError(f"Missing TRAIN_SAVE_DIR: {save_dir}")

script_path = PROJECT_ROOT / "phycausal_stgrn" / "scripts" / "rebuild_main_claim_summaries.py"
if not script_path.exists():
    raise FileNotFoundError(f"Missing helper script: {script_path}")

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONUTF8"] = "1"
env["PYTHONIOENCODING"] = "utf-8"

cmd = [
    sys.executable,
    str(script_path),
    "--config", str(CFG_PATH),
    "--save_dir", str(save_dir),
    "--edge_method", "jacobian",
    "--top_k", "2000",
]
print("🧪 [Module3.5] Running:", " ".join(cmd))
rc = subprocess.call(cmd, cwd=str(PROJECT_ROOT), env=env)
if rc != 0:
    raise RuntimeError(f"rebuild_main_claim_summaries.py failed with return code {rc}")

for name in ["paper_summary.json", "counterfactual_summary.json", "gate_summary.json"]:
    p = save_dir / name
    print(("✅" if p.exists() else "❌"), p)
